# 아기캥거루 Kaggle 제출 파이프라인 V3.1

`IntegratedDataset.tar`의 inference 이미지를 전처리하고, 앞 단계에서 선택한 YOLO11s / YOLO11m / YOLO12m 체크포인트로 추론해 Kaggle 제출 CSV를 생성한다.

전처리는 학습 데이터와 같은 순서를 사용하며, ICC 프로필이 있는 이미지는 sRGB로 변환한 뒤 알파 채널을 합성한다. 이미 sRGB인 경우 불필요한 프로필 변환은 생략한다.

이 노트북에서는 재학습하지 않는다.

## 실행 전 준비

아래 산출물이 준비되어 있어야 한다.

- `baby_kangaroo_cache/IntegratedDataset.tar`
- `baby_kangaroo_cache/IntegratedDataset/dataset_summary.json`
- `baby_kangaroo_cache/IntegratedDataset/split_manifest.csv`
- `week2/공통파이프라인/checkpoints/*_best.pt`
- `week2/공통파이프라인/manifests/*.json`
- `week2/공통파이프라인/reports/kaggle_domain_eval_all9_*_summary.json`

먼저 `RUN_MODE="smoke"`로 일부 이미지를 확인한 뒤, 전체 제출 생성 시 `RUN_MODE="full"`을 사용한다.

# 1. 실행 환경

## 1-1. 패키지 설치

PyTorch import 전에 메모리 관련 환경 변수를 설정하고, 체크포인트와 호환되는 패키지 버전을 사용한다.

## 1-2. 라이브러리, Drive, 환경 정보

In [16]:
# Colab 기본 PyTorch/CUDA 조합은 유지하고 필요한 상위 패키지만 고정한다.
import os
import subprocess
import sys

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

print("[Pipeline 01/14 |   7%] START: Package installation")

PINNED_PACKAGES = [
    "ultralytics==8.4.116",
    "tqdm==4.67.1",
]

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", *PINNED_PACKAGES
])

print("Package installation completed.")
print("PyTorch/torchvision were NOT force-downgraded; Colab runtime CUDA compatibility is preserved.")
print("[Pipeline 01/14 |   7%] DONE: Package installation")


[Pipeline 01/14 |   7%] START: Package installation
Package installation completed.
PyTorch/torchvision were NOT force-downgraded; Colab runtime CUDA compatibility is preserved.
[Pipeline 01/14 |   7%] DONE: Package installation


In [17]:
# 파일 검증, 이미지 전처리, 모델 추론과 결과 저장에 필요한 라이브러리를 불러온다.
import gc
import hashlib
import importlib.metadata as importlib_metadata
import json
import math
import os
import random
import re
import shutil
import sys
import time
import tarfile
from datetime import datetime
from io import BytesIO
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torchvision
from PIL import Image, ImageCms, ImageDraw, ImageOps
# [삭제] FRCNN용 불필요한 torch.utils.data 및 torchvision.transforms 임포트 제거
from tqdm.auto import tqdm
from ultralytics import YOLO


PIPELINE_STAGE_TOTAL = 14


def show_stage_progress(stage, title, status):
    """주요 셀의 전체 파이프라인 진행률을 같은 형식으로 표시한다."""
    percent = int(round(stage / PIPELINE_STAGE_TOTAL * 100))
    print(
        f"[Pipeline {stage:02d}/{PIPELINE_STAGE_TOTAL} | {percent:3d}%] "
        f"{status}: {title}"
    )


show_stage_progress(2, "Runtime, libraries, and Google Drive", "START")

try:
    from google.colab import drive
except ImportError as error:
    raise RuntimeError("This notebook must be executed in Google Colab.") from error

drive.mount("/content/drive")

# 실행할 때마다 같은 입력 순서와 결과가 나오도록 난수를 고정한다.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TORCH_CUDA_COMPILED = torch.version.cuda is not None
CUDA_AVAILABLE = torch.cuda.is_available()
RUNTIME_VERSIONS = {
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "ultralytics": importlib_metadata.version("ultralytics"),
    "opencv": cv2.__version__,
    "pillow": importlib_metadata.version("Pillow"),
    "pandas": pd.__version__,
    "numpy": np.__version__,
}

print(f"Device: {DEVICE}")
print(f"Torch CUDA compiled: {TORCH_CUDA_COMPILED}")
print(f"CUDA available now: {CUDA_AVAILABLE}")
print(json.dumps(RUNTIME_VERSIONS, indent=2))
show_stage_progress(2, "Runtime, libraries, and Google Drive", "DONE")

[Pipeline 02/14 |  14%] START: Runtime, libraries, and Google Drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda
Torch CUDA compiled: True
CUDA available now: True
{
  "python": "3.12.13",
  "torch": "2.11.0+cu128",
  "torchvision": "0.26.0+cu128",
  "ultralytics": "8.4.116",
  "opencv": "4.14.0",
  "pillow": "11.3.0",
  "pandas": "2.2.3",
  "numpy": "2.0.2"
}
[Pipeline 02/14 |  14%] DONE: Runtime, libraries, and Google Drive


## 1-3. 공통 설정과 경로

입력 아카이브는 `IntegratedDataset.tar` 하나로 고정한다. 파일 존재 여부와 `inference_images` 842장을 확인한 뒤 다음 단계로 진행한다.

In [18]:
show_stage_progress(3, "IntegratedDataset.tar V3.0 configuration and paths", "START")

RUN_MODE = "full"  # "smoke" 또는 "full"
SMOKE_IMAGE_LIMIT = 8

CANDIDATE_MODELS = ("YOLO11s", "YOLO11m", "YOLO12m")
MODELS_TO_SUBMIT = CANDIDATE_MODELS

KAGGLE_DOMAIN_SUMMARY_OVERRIDE = None
SAMPLE_SUBMISSION_PATH_OVERRIDE = None
USE_TTA = False
ENABLE_INFERENCE_CACHE = True

COMPETITION_METRIC = "mAP@[0.75:0.95]"
KAGGLE_SUBMISSION_COLUMNS = [
    "annotation_id", "image_id", "category_id",
    "bbox_x", "bbox_y", "bbox_w", "bbox_h", "score",
]

PIPELINE_VERSION = "3.0-integrated-tar-kaggle-domain"

EXPECTED_MODEL_TOTAL_IMAGES = 12196
EXPECTED_MODEL_OBJECTS = 46394
EXPECTED_MODEL_CLASSES = 118
EXPECTED_MODEL_GROUPS = 4104
EXPECTED_MODEL_TRAIN_IMAGES = 9763
EXPECTED_MODEL_VAL_IMAGES = 2433
EXPECTED_KAGGLE_TEST_IMAGES = 842

EXPECTED_BASE_IMAGES = 232
EXPECTED_BASE_TRAIN_IMAGES = 176
EXPECTED_BASE_VAL_IMAGES = 56

KAGGLE_OFFICIAL_CLASS_COUNT = 56
KAGGLE_OFFICIAL_YOLO_CLASS_IDS = tuple(range(KAGGLE_OFFICIAL_CLASS_COUNT))
EXPECTED_SPLIT_FINGERPRINT_PREFIX = "cc5d16d3fe042c52"
EXPECTED_CLASSES = EXPECTED_MODEL_CLASSES

IMAGE_SIZE = 960
RAW_PREDICTION_CONFIDENCE = 0.001
NMS_IOU_THRESHOLD = 0.70
MAX_DETECTIONS = 300
MAX_PILLS_PER_IMAGE = None
YOLO_INFERENCE_BATCH_SIZE = 8
VISUALIZATION_IMAGE_COUNT = 4
SUPPORTED_IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"
}

if RUN_MODE not in {"smoke", "full"}:
    raise ValueError("RUN_MODE must be 'smoke' or 'full'.")
if RUN_MODE == "full" and not torch.cuda.is_available():
    raise RuntimeError("Full Kaggle inference requires a GPU runtime.")

BASE_ROOT = Path("/content/drive/MyDrive/baby_kangaroo")
SHARED_ROOT = BASE_ROOT / "week2"
COMMON_CACHE_ROOT = BASE_ROOT / "baby_kangaroo_cache"

# 정확히 이 TAR 하나만 사용한다. fallback 금지.
INTEGRATED_DATASET_TAR = COMMON_CACHE_ROOT / "IntegratedDataset.tar"
INTEGRATED_DATASET_ROOT = COMMON_CACHE_ROOT / "IntegratedDataset"
INTEGRATED_SUMMARY_PATH = INTEGRATED_DATASET_ROOT / "dataset_summary.json"
INTEGRATED_SPLIT_MANIFEST_PATH = INTEGRATED_DATASET_ROOT / "split_manifest.csv"

PREPROCESS_ROOT = (
    SHARED_ROOT / "데이터전처리" / "yolo_전처리"
    / "pill_yolo_full_v6_0_preprocessed"
)
PREPROCESS_CONFIG_PATH = PREPROCESS_ROOT / "preprocessing_config.json"
ARTIFACT_ROOT = SHARED_ROOT / "공통파이프라인"
CHECKPOINT_DIR = ARTIFACT_ROOT / "checkpoints"
MODEL_MANIFEST_DIR = ARTIFACT_ROOT / "manifests"
EVAL_REPORT_DIR = ARTIFACT_ROOT / "reports"

YOLO_BUNDLE_DIRS = {
    model_name: PREPROCESS_ROOT
    for model_name in CANDIDATE_MODELS
}

DRIVE_KAGGLE_ROOT = SHARED_ROOT / "kaggle_submission_v3_integrated"

LOCAL_WORK_ROOT = Path("/content/baby_kangaroo_kaggle_submission_v30")
LOCAL_INPUT_ROOT = LOCAL_WORK_ROOT / "input"
LOCAL_MODEL_INPUT_DIR = LOCAL_WORK_ROOT / "model_input"
LOCAL_RESULT_ROOT = LOCAL_WORK_ROOT / "results"
KAGGLE_TEST_IMAGE_DIR = LOCAL_INPUT_ROOT / "inference_images"

FINAL_CONFIG_DIR = LOCAL_RESULT_ROOT / "final_config"
PREPROCESSED_ROOT = LOCAL_RESULT_ROOT / "preprocessed_test"
PREDICTION_DIR = LOCAL_RESULT_ROOT / "predictions"
SUBMISSION_DIR = LOCAL_RESULT_ROOT / "submissions"
FIGURE_DIR = LOCAL_RESULT_ROOT / "figures"
REPORT_DIR = LOCAL_RESULT_ROOT / "reports"

DRIVE_FINAL_CONFIG_DIR = DRIVE_KAGGLE_ROOT / "final_config"
DRIVE_PREDICTION_DIR = DRIVE_KAGGLE_ROOT / "predictions"
DRIVE_SUBMISSION_DIR = DRIVE_KAGGLE_ROOT / "submissions"
DRIVE_FIGURE_DIR = DRIVE_KAGGLE_ROOT / "figures"
DRIVE_REPORT_DIR = DRIVE_KAGGLE_ROOT / "reports"
DRIVE_PREPROCESSING_MANIFEST_DIR = DRIVE_KAGGLE_ROOT / "preprocessing_manifests"

for directory in [
    LOCAL_INPUT_ROOT, LOCAL_MODEL_INPUT_DIR,
    FINAL_CONFIG_DIR, PREPROCESSED_ROOT, PREDICTION_DIR,
    SUBMISSION_DIR, FIGURE_DIR, REPORT_DIR,
    DRIVE_FINAL_CONFIG_DIR, DRIVE_PREDICTION_DIR,
    DRIVE_SUBMISSION_DIR, DRIVE_FIGURE_DIR,
    DRIVE_REPORT_DIR, DRIVE_PREPROCESSING_MANIFEST_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

for required_path in [
    INTEGRATED_DATASET_TAR,
    INTEGRATED_SUMMARY_PATH,
    INTEGRATED_SPLIT_MANIFEST_PATH,
    PREPROCESS_ROOT / "class_mapping.csv",
    PREPROCESS_CONFIG_PATH,
    PREPROCESS_ROOT / "dataset_manifest.csv",
    CHECKPOINT_DIR,
    MODEL_MANIFEST_DIR,
    EVAL_REPORT_DIR,
]:
    if not required_path.exists():
        raise FileNotFoundError(f"Required current resource missing: {required_path}")

# IntegratedDataset contract
integrated_summary = json.loads(INTEGRATED_SUMMARY_PATH.read_text(encoding="utf-8"))
expected_integrated = {
    "num_classes": EXPECTED_MODEL_CLASSES,
    "train_images": EXPECTED_MODEL_TRAIN_IMAGES,
    "val_images": EXPECTED_MODEL_VAL_IMAGES,
    "total_labeled_images": EXPECTED_MODEL_TOTAL_IMAGES,
    "total_objects": EXPECTED_MODEL_OBJECTS,
    "inference_images": EXPECTED_KAGGLE_TEST_IMAGES,
}
for key, expected in expected_integrated.items():
    actual = int(integrated_summary.get(key, -1))
    if actual != expected:
        raise RuntimeError(
            f"IntegratedDataset summary mismatch: {key}={actual}, expected={expected}"
        )

integrated_split_df = pd.read_csv(INTEGRATED_SPLIT_MANIFEST_PATH)
if len(integrated_split_df) != EXPECTED_MODEL_TOTAL_IMAGES:
    raise RuntimeError("IntegratedDataset split_manifest row count mismatch.")

source_split_counts = integrated_split_df.groupby(["source", "split"]).size().to_dict()
expected_source_split_counts = {
    ("base", "train"): EXPECTED_BASE_TRAIN_IMAGES,
    ("base", "val"): EXPECTED_BASE_VAL_IMAGES,
    ("extra", "train"): EXPECTED_MODEL_TRAIN_IMAGES - EXPECTED_BASE_TRAIN_IMAGES,
    ("extra", "val"): EXPECTED_MODEL_VAL_IMAGES - EXPECTED_BASE_VAL_IMAGES,
}
if source_split_counts != expected_source_split_counts:
    raise RuntimeError(
        f"IntegratedDataset source/split contract mismatch: {source_split_counts}"
    )

base_class_ids = sorted({
    int(token)
    for value in integrated_split_df.loc[
        integrated_split_df["source"].eq("base"), "class_ids"
    ].dropna().astype(str)
    for token in value.split(",")
    if token.strip()
})
if base_class_ids != list(KAGGLE_OFFICIAL_YOLO_CLASS_IDS):
    raise RuntimeError(
        f"Official Kaggle class contract must be model IDs 0..55, got {base_class_ids}"
    )

pipeline_timer_started_at = datetime.now()
pipeline_timer_started_perf = time.perf_counter()

print(f"Run mode: {RUN_MODE}")
print(f"IntegratedDataset.tar ONLY: {INTEGRATED_DATASET_TAR}")
print(f"IntegratedDataset: {EXPECTED_MODEL_TOTAL_IMAGES:,} labeled / {EXPECTED_MODEL_CLASSES} classes / inference {EXPECTED_KAGGLE_TEST_IMAGES}")
print(f"Base source: {EXPECTED_BASE_IMAGES} = train {EXPECTED_BASE_TRAIN_IMAGES} + val {EXPECTED_BASE_VAL_IMAGES}")
print(f"Current preprocessing root: {PREPROCESS_ROOT}")
print(f"Kaggle-domain evaluation reports: {EVAL_REPORT_DIR}")
print(f"Output root: {DRIVE_KAGGLE_ROOT}")

show_stage_progress(3, "IntegratedDataset.tar V3.0 configuration and paths", "DONE")


[Pipeline 03/14 |  21%] START: IntegratedDataset.tar V3.0 configuration and paths
Run mode: full
IntegratedDataset.tar ONLY: /content/drive/MyDrive/baby_kangaroo/baby_kangaroo_cache/IntegratedDataset.tar
IntegratedDataset: 12,196 labeled / 118 classes / inference 842
Base source: 232 = train 176 + val 56
Current preprocessing root: /content/drive/MyDrive/baby_kangaroo/week2/데이터전처리/yolo_전처리/pill_yolo_full_v6_0_preprocessed
Kaggle-domain evaluation reports: /content/drive/MyDrive/baby_kangaroo/week2/공통파이프라인/reports
Output root: /content/drive/MyDrive/baby_kangaroo/week2/kaggle_submission_v3_integrated
[Pipeline 03/14 |  21%] DONE: IntegratedDataset.tar V3.0 configuration and paths


`IntegratedDataset.tar`와 Kaggle-domain 평가 summary를 기준으로 제출 설정을 구성한다.

# 2. 모델 설정 불러오기

모델 개발 파이프라인의 `standard` 보고서에서 YOLO11s / YOLO11m / YOLO12m의 선택된 체크포인트와 Validation confidence를 읽는다. 체크포인트 해시도 함께 확인한다.

In [19]:
show_stage_progress(4, "Reusable validation and storage helpers", "START")

def sha256_file(path, chunk_size=1024 * 1024):
    """파일을 메모리에 한꺼번에 올리지 않고 SHA256 지문을 계산한다."""
    digest = hashlib.sha256()
    with Path(path).open("rb") as file:
        while True:
            chunk = file.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def stable_json_hash(value):
    """딕셔너리 순서와 관계없이 같은 JSON 내용에 같은 SHA256 지문을 만든다."""
    encoded = json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    ).encode("utf-8")
    return hashlib.sha256(encoded).hexdigest()


def json_ready(value):
    """Path·NumPy 자료형을 JSON으로 저장 가능한 기본 자료형으로 바꾼다."""
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {str(key): json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(item) for item in value]
    if isinstance(value, np.generic):
        return value.item()
    return value


def write_json_atomic(path, value):
    """완성되지 않은 JSON이 남지 않도록 임시 파일을 거쳐 원자적으로 저장한다."""
    path = Path(path)
    temporary_path = path.with_suffix(path.suffix + ".tmp")
    temporary_path.write_text(
        json.dumps(json_ready(value), ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    temporary_path.replace(path)


def find_latest_standard_report(report_dir, expected_pipeline_version=None):
    """가장 최근에 완주한 standard 모델개발 보고서를 찾고 파이프라인 버전 호환성을 종합 검증한다."""
    candidates = sorted(
        Path(report_dir).glob("model_development_report_*.json"),
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )
    for path in candidates:
        try:
            report = json.loads(path.read_text(encoding="utf-8"))
        except (OSError, json.JSONDecodeError):
            continue
        if report.get("run_profile") == "standard":
            # pipeline version과 데이터 계약의 호환성을 확인한다.
            if expected_pipeline_version and report.get("pipeline_version") != expected_pipeline_version:
                continue
            return path, report

    raise FileNotFoundError(
        "No completed standard model development report matching the required pipeline version was found. "
        "Run the model development notebook in standard mode first."
    )

def find_checkpoint_manifest(checkpoint_path, model_name=None):
    """선택된 checkpoint 경로/모델 이름에 대응하는 완료 manifest를 찾아 해시와 내용을 엄격히 대조 검증한다."""
    checkpoint_path = Path(checkpoint_path)
    actual_hash = sha256_file(checkpoint_path)

    # 모델명이 직접 주어진 경우 또는 파일명에서 베이스 이름을 추출
    if model_name:
        target_prefix = model_name
    else:
        target_prefix = re.sub(r"(_best|_last)?\.(pth|pt)$", "", checkpoint_path.name)

    # 앙상블 대응: 단일 파일 검색 및 하위 디렉터리 재귀 검색 지원
    candidates = sorted(MODEL_MANIFEST_DIR.rglob(f"*{target_prefix}*.json"))

    # 모델 이름과 파일명이 일치하는 manifest 후보를 찾는다.
    valid_candidates = []
    for cand in candidates:
        try:
            m_data = json.loads(cand.read_text(encoding="utf-8"))
            if m_data.get("model_name") == model_name or target_prefix in cand.name:
                valid_candidates.append((cand, m_data))
        except (OSError, json.JSONDecodeError):
            continue

    if not valid_candidates:
        raise FileNotFoundError(
            f"Expected checkpoint manifest for {checkpoint_path.name} (model: {model_name}), but found none in {MODEL_MANIFEST_DIR}."
        )

    # 체크포인트 SHA-256과 일치하는 manifest를 우선 선택한다.
    matched_pair = None
    for cand, m_data in valid_candidates:
        expected_hash = m_data.get("checkpoint_sha256") or m_data.get("weights_sha256")
        if expected_hash and expected_hash == actual_hash:
            matched_pair = (cand, m_data)
            break

    # 해시 일치 항목이 없으면 최신 후보를 선택하고 아래 검증에서 불일치를 보고한다.
    if matched_pair is None:
        manifest_path, manifest = sorted(valid_candidates, key=lambda x: x[0].stat().st_mtime, reverse=True)[0]
    else:
        manifest_path, manifest = matched_pair

    # training_completed가 명시적으로 True인지 확인한다.
    if manifest.get("training_completed") is not True:
        raise RuntimeError(f"Training not completed or manifest incomplete for {checkpoint_path}")

    # 체크포인트 SHA-256이 manifest와 일치하는지 확인한다.
    expected_hash = manifest.get("checkpoint_sha256") or manifest.get("weights_sha256")
    if expected_hash and expected_hash != actual_hash:
        raise RuntimeError(f"Checkpoint SHA256 mismatch for {checkpoint_path.name}: expected {expected_hash}, got {actual_hash}")

    return manifest_path, manifest, actual_hash


def list_supported_images(image_dir):
    """지원 확장자의 이미지를 재귀적으로 수집한다."""
    image_dir = Path(image_dir)
    if not image_dir.is_dir():
        raise FileNotFoundError(f"Image directory not found: {image_dir}")
    return sorted(
        path for path in image_dir.rglob("*")
        if path.is_file() and path.suffix.lower() in SUPPORTED_IMAGE_EXTENSIONS
    )

def validate_test_image_list(image_paths, source_name, expected_count=None):
    """Kaggle Test 이미지 수와 파일명 충돌을 동적으로 검사한다."""
    actual_count = len(image_paths)

    if expected_count is None:
        expected_count = EXPECTED_KAGGLE_TEST_IMAGES
    if expected_count is not None and actual_count != int(expected_count):
        raise RuntimeError(
            f"Expected {expected_count} staged Kaggle images in {source_name}, "
            f"found {actual_count}."
        )

    # 처리할 이미지가 없으면 실행을 중단한다.
    if actual_count == 0:
        raise RuntimeError(f"No test images found in {source_name}.")

    basenames = [path.name for path in image_paths]
    stems = [path.stem for path in image_paths]
    if len(basenames) != len(set(basenames)):
        raise RuntimeError(f"Duplicate test image basenames were found in {source_name}.")
    if len(stems) != len(set(stems)):
        raise RuntimeError(
            f"Test images with identical stems and different extensions were found in {source_name}."
        )
    return image_paths


def reset_local_owned_directory(directory):
    """이 노트북이 소유한 /content 하위 폴더만 안전하게 초기화한다."""
    directory = Path(directory).resolve()
    owned_root = LOCAL_WORK_ROOT.resolve()
    if directory == owned_root or owned_root not in directory.parents:
        raise RuntimeError(f"Local directory reset is not allowed: {directory}")
    if directory.exists():
        shutil.rmtree(directory)
    directory.mkdir(parents=True, exist_ok=True)


def copy_file_verified(source, destination, expected_sha256=None):
    """파일을 임시 이름으로 복사한 뒤 크기·선택적 SHA256을 확인해 교체한다."""
    source = Path(source)
    destination = Path(destination)
    if not source.is_file():
        raise FileNotFoundError(f"Source file not found: {source}")
    destination.parent.mkdir(parents=True, exist_ok=True)

    source_stat = source.stat()
    reusable = destination.is_file() and destination.stat().st_size == source_stat.st_size
    if reusable and expected_sha256 is not None:
        reusable = sha256_file(destination) == expected_sha256
    elif reusable:
        reusable = int(destination.stat().st_mtime) == int(source_stat.st_mtime)
    if reusable:
        print(f"Reused verified local file: {destination}")
        return destination

    temporary_path = destination.with_suffix(destination.suffix + ".copying")
    shutil.copy2(source, temporary_path)
    if temporary_path.stat().st_size != source_stat.st_size:
        temporary_path.unlink(missing_ok=True)
        raise RuntimeError(f"Copied file size mismatch: {source} -> {destination}")
    if expected_sha256 is not None and sha256_file(temporary_path) != expected_sha256:
        temporary_path.unlink(missing_ok=True)
        raise RuntimeError(f"Copied file SHA256 mismatch: {source} -> {destination}")
    temporary_path.replace(destination)
    print(f"Copied to Colab local storage: {source.name}")
    return destination


def _force_remount_drive_for_tar():
    """Drive FUSE I/O 오류 시 TAR 읽기를 처음부터 다시 시도하기 위한 remount."""
    try:
        drive.flush_and_unmount()
    except Exception:
        pass
    time.sleep(1.0)
    drive.mount("/content/drive", force_remount=True)
    time.sleep(2.0)


def _scan_integrated_inference_members(archive_path):
    """IntegratedDataset.tar에서 inference_images의 지원 이미지 842개만 식별한다."""
    with tarfile.open(archive_path, "r:*") as archive:
        members = []
        for member in archive.getmembers():
            if not member.isfile():
                continue
            parts = [part for part in member.name.replace("\\", "/").split("/") if part]
            if "inference_images" not in parts:
                continue
            suffix = Path(parts[-1]).suffix.lower()
            if suffix in SUPPORTED_IMAGE_EXTENSIONS:
                members.append(member)

    basenames = [Path(m.name).name for m in members]
    stems = [Path(name).stem for name in basenames]
    if len(members) != EXPECTED_KAGGLE_TEST_IMAGES:
        raise RuntimeError(
            f"IntegratedDataset.tar must contain exactly {EXPECTED_KAGGLE_TEST_IMAGES} "
            f"inference_images, found {len(members)}."
        )
    if len(basenames) != len(set(basenames)):
        raise RuntimeError("Duplicate inference image basenames inside IntegratedDataset.tar.")
    if len(stems) != len(set(stems)):
        raise RuntimeError("Duplicate inference image stems inside IntegratedDataset.tar.")
    return sorted(members, key=lambda m: Path(m.name).name)


def stage_kaggle_inputs_to_local():
    """IntegratedDataset.tar에서 inference_images 842장만 /content로 선택 추출한다."""
    archive_path = INTEGRATED_DATASET_TAR
    if not archive_path.is_file():
        raise FileNotFoundError(
            f"IntegratedDataset.tar is required and no fallback is allowed: {archive_path}"
        )

    extract_dir = KAGGLE_TEST_IMAGE_DIR
    marker_path = LOCAL_INPUT_ROOT / "integrated_tar_staging_contract.json"

    last_error = None
    for attempt in range(1, 4):
        try:
            all_members = _scan_integrated_inference_members(archive_path)
            members = (
                all_members[:SMOKE_IMAGE_LIMIT]
                if RUN_MODE == "smoke"
                else all_members
            )

            member_contract = [
                {
                    "archive_member": m.name,
                    "file_name": Path(m.name).name,
                    "size_bytes": int(m.size),
                }
                for m in members
            ]
            archive_contract = {
                "source_path": str(archive_path),
                "source_size_bytes": int(archive_path.stat().st_size),
                "source_mtime": int(archive_path.stat().st_mtime),
                "inference_images_scanned": EXPECTED_KAGGLE_TEST_IMAGES,
                "inference_images_extracted": len(member_contract),
                "run_mode": RUN_MODE,
                "member_index_sha256": stable_json_hash(member_contract),
            }

            reuse = False
            if marker_path.is_file() and extract_dir.is_dir():
                try:
                    saved = json.loads(marker_path.read_text(encoding="utf-8"))
                    current_files = list_supported_images(extract_dir)
                    reuse = (
                        saved == archive_contract
                        and len(current_files) == len(member_contract)
                        and {p.name for p in current_files}
                            == {row["file_name"] for row in member_contract}
                    )
                except (OSError, json.JSONDecodeError):
                    reuse = False

            if not reuse:
                reset_local_owned_directory(extract_dir)
                required_bytes = sum(int(m.size) for m in members)
                available_bytes = shutil.disk_usage(LOCAL_WORK_ROOT).free
                if required_bytes > available_bytes * 0.80:
                    raise RuntimeError(
                        f"Not enough local disk for 842 inference images: "
                        f"need={required_bytes}, free={available_bytes}"
                    )

                with tarfile.open(archive_path, "r:*") as archive:
                    member_map = {m.name: m for m in archive.getmembers()}
                    for row in tqdm(
                        member_contract,
                        desc="Extract IntegratedDataset inference_images",
                        unit="image",
                    ):
                        member = member_map[row["archive_member"]]
                        source = archive.extractfile(member)
                        if source is None:
                            raise RuntimeError(f"Could not read TAR member: {member.name}")

                        destination = extract_dir / row["file_name"]
                        temporary = destination.with_suffix(destination.suffix + ".part")
                        with temporary.open("wb") as target:
                            shutil.copyfileobj(source, target, length=1024 * 1024)

                        if temporary.stat().st_size != int(member.size):
                            temporary.unlink(missing_ok=True)
                            raise RuntimeError(
                                f"Extracted size mismatch: {member.name}"
                            )
                        os.replace(temporary, destination)

                write_json_atomic(marker_path, archive_contract)
                print("IntegratedDataset.tar inference_images extracted.")
            else:
                print("Reused verified IntegratedDataset.tar inference_images cache.")

            expected_staged_count = (
                SMOKE_IMAGE_LIMIT if RUN_MODE == "smoke"
                else EXPECTED_KAGGLE_TEST_IMAGES
            )
            test_images = validate_test_image_list(
                list_supported_images(extract_dir),
                str(extract_dir),
                expected_count=expected_staged_count,
            )
            staging_rows = []
            size_by_name = {row["file_name"]: row for row in member_contract}
            for path in test_images:
                meta = size_by_name[path.name]
                staging_rows.append({
                    "file_name": path.name,
                    "local_path": str(path),
                    "size_bytes": int(path.stat().st_size),
                    "archive_member": meta["archive_member"],
                    "staging_mode": "integrated_dataset_tar_selective_inference_extract",
                    "drive_source": str(archive_path),
                    "archive_size_bytes": int(archive_path.stat().st_size),
                    "archive_member_index_sha256": archive_contract["member_index_sha256"],
                })

            return (
                extract_dir,
                "integrated_dataset_tar_selective_inference_extract",
                pd.DataFrame(staging_rows),
            )

        except (OSError, IOError) as exc:
            last_error = exc
            print(
                f"[IntegratedDataset.tar retry {attempt}/3] "
                f"{type(exc).__name__}: {exc}"
            )
            if attempt >= 3:
                break
            _force_remount_drive_for_tar()

    raise RuntimeError(
        "IntegratedDataset.tar selective extraction failed after 3 attempts."
    ) from last_error


def persist_artifact_to_drive(local_path, drive_directory):
    """작은 최종 산출물만 Drive에 복사하고 SHA256 동일성을 확인한다."""
    local_path = Path(local_path)
    drive_directory = Path(drive_directory)
    drive_directory.mkdir(parents=True, exist_ok=True)
    destination = drive_directory / local_path.name
    expected_hash = sha256_file(local_path)
    return copy_file_verified(local_path, destination, expected_sha256=expected_hash)

show_stage_progress(4, "IntegratedDataset.tar validation and storage helpers", "DONE")

[Pipeline 04/14 |  29%] START: Reusable validation and storage helpers
[Pipeline 04/14 |  29%] DONE: IntegratedDataset.tar validation and storage helpers


In [20]:
show_stage_progress(5, "Load Kaggle-domain all-9 evaluation summary", "START")

def find_latest_kaggle_domain_summary():
    if KAGGLE_DOMAIN_SUMMARY_OVERRIDE:
        path = Path(KAGGLE_DOMAIN_SUMMARY_OVERRIDE)
        if not path.is_file():
            raise FileNotFoundError(f"Kaggle-domain summary override not found: {path}")
        return path

    candidates = sorted(
        EVAL_REPORT_DIR.glob(
            f"kaggle_domain_eval_all9_{EXPECTED_SPLIT_FINGERPRINT_PREFIX}_*_summary.json"
        ),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    if not candidates:
        raise FileNotFoundError(
            "No Kaggle-domain all-9 summary found. "
            "Run Kaggle_Domain_EVAL_V5_BASE56_ALL9.ipynb first."
        )
    return candidates[0]

summary_path = find_latest_kaggle_domain_summary()
domain_summary = json.loads(summary_path.read_text(encoding="utf-8"))

required_summary_keys = {
    "mode", "training_disabled", "selection_scope", "competition_metric",
    "split_policy", "split_fingerprint", "integrated_dataset_contract",
    "kaggle_domain_contract", "architecture_winners", "recommended", "ranking",
}
missing = required_summary_keys - set(domain_summary)
if missing:
    raise KeyError(f"Kaggle-domain summary missing keys: {sorted(missing)}")

if domain_summary["mode"] != "kaggle_domain_evaluation_only":
    raise RuntimeError(f"Unexpected evaluation mode: {domain_summary['mode']}")
if domain_summary["training_disabled"] is not True:
    raise RuntimeError("Kaggle-domain summary is not evaluation-only.")
if domain_summary["selection_scope"] != "all_9_completed_checkpoints":
    raise RuntimeError("Kaggle-domain summary did not compare all 9 checkpoints.")
if domain_summary["competition_metric"] != COMPETITION_METRIC:
    raise RuntimeError("Kaggle-domain metric mismatch.")
if domain_summary["split_policy"] != "preserve_exact_upstream":
    raise RuntimeError("Kaggle-domain split policy mismatch.")
if not str(domain_summary["split_fingerprint"]).startswith(
    EXPECTED_SPLIT_FINGERPRINT_PREFIX
):
    raise RuntimeError("Kaggle-domain split fingerprint mismatch.")

dataset_contract = domain_summary["integrated_dataset_contract"]
expected_contract = {
    "total_labeled_images": EXPECTED_MODEL_TOTAL_IMAGES,
    "train_images": EXPECTED_MODEL_TRAIN_IMAGES,
    "validation_images": EXPECTED_MODEL_VAL_IMAGES,
    "objects": EXPECTED_MODEL_OBJECTS,
    "classes": EXPECTED_MODEL_CLASSES,
    "inference_images": EXPECTED_KAGGLE_TEST_IMAGES,
}
for key, expected in expected_contract.items():
    if int(dataset_contract.get(key, -1)) != expected:
        raise RuntimeError(f"Kaggle-domain dataset contract mismatch: {key}")

domain_contract = domain_summary["kaggle_domain_contract"]
if domain_contract.get("source") != "base":
    raise RuntimeError("Kaggle-domain summary source must be 'base'.")
if int(domain_contract.get("base_validation_images", -1)) != EXPECTED_BASE_VAL_IMAGES:
    raise RuntimeError("Kaggle-domain Validation must contain exactly 56 base images.")
if int(domain_contract.get("official_class_count", -1)) != KAGGLE_OFFICIAL_CLASS_COUNT:
    raise RuntimeError("Kaggle-domain official class count mismatch.")
if list(map(int, domain_contract.get("official_model_class_ids", []))) != list(
    KAGGLE_OFFICIAL_YOLO_CLASS_IDS
):
    raise RuntimeError("Kaggle-domain official model class IDs changed.")

architecture_winners = domain_summary["architecture_winners"]
if set(architecture_winners) != set(CANDIDATE_MODELS):
    raise RuntimeError(
        f"Architecture winners must contain {CANDIDATE_MODELS}, got {architecture_winners.keys()}"
    )

MODEL_RUN_CONFIGS = {}
selection_rows = []

for model_name in CANDIDATE_MODELS:
    winner = architecture_winners[model_name]
    experiment_id = str(winner["experiment_id"])
    stage = str(winner["stage"])

    expected_checkpoint_name = f"{experiment_id}_best.pt"
    source_checkpoint_path = CHECKPOINT_DIR / expected_checkpoint_name
    source_manifest_path = MODEL_MANIFEST_DIR / f"{experiment_id}.json"

    if not source_checkpoint_path.is_file():
        raise FileNotFoundError(source_checkpoint_path)
    if not source_manifest_path.is_file():
        raise FileNotFoundError(source_manifest_path)

    manifest = json.loads(source_manifest_path.read_text(encoding="utf-8"))
    if manifest.get("training_completed") is not True:
        raise RuntimeError(f"{model_name} checkpoint is not training_completed.")
    if manifest.get("model_name") != model_name:
        raise RuntimeError(f"{model_name} manifest model mismatch.")
    if manifest.get("split_fingerprint") != domain_summary["split_fingerprint"]:
        raise RuntimeError(f"{model_name} split fingerprint mismatch.")

    split_config = manifest.get("split_config", {})
    manifest_checks = {
        "expected_images": EXPECTED_MODEL_TOTAL_IMAGES,
        "expected_objects": EXPECTED_MODEL_OBJECTS,
        "expected_num_classes": EXPECTED_MODEL_CLASSES,
        "expected_groups": EXPECTED_MODEL_GROUPS,
    }
    for key, expected in manifest_checks.items():
        if int(split_config.get(key, -1)) != expected:
            raise RuntimeError(f"{model_name} manifest contract mismatch: {key}")

    checkpoint_sha256 = sha256_file(source_checkpoint_path)
    expected_hash = manifest.get("checkpoint_sha256")
    if expected_hash and checkpoint_sha256 != expected_hash:
        raise RuntimeError(f"{model_name} checkpoint SHA256 mismatch.")

    local_summary_path = LOCAL_MODEL_INPUT_DIR / summary_path.name
    local_manifest_path = LOCAL_MODEL_INPUT_DIR / f"{model_name}_{source_manifest_path.name}"
    local_checkpoint_path = (
        LOCAL_MODEL_INPUT_DIR
        / f"{model_name}_{checkpoint_sha256[:12]}_{source_checkpoint_path.name}"
    )

    copy_file_verified(summary_path, local_summary_path, expected_sha256=sha256_file(summary_path))
    copy_file_verified(
        source_manifest_path,
        local_manifest_path,
        expected_sha256=sha256_file(source_manifest_path),
    )
    copy_file_verified(
        source_checkpoint_path,
        local_checkpoint_path,
        expected_sha256=checkpoint_sha256,
    )

    selected_top_k = (
        None if winner.get("selected_top_k") is None
        or pd.isna(winner.get("selected_top_k"))
        else int(winner["selected_top_k"])
    )

    MODEL_RUN_CONFIGS[model_name] = {
        "model_name": model_name,
        "experiment_id": experiment_id,
        "selected_stage": stage,
        "source_summary_path": summary_path,
        "local_summary_path": local_summary_path,
        "source_checkpoint_path": source_checkpoint_path,
        "local_checkpoint_path": local_checkpoint_path,
        "checkpoint_sha256": checkpoint_sha256,
        "source_manifest_path": source_manifest_path,
        "local_manifest_path": local_manifest_path,
        "selected_confidence": float(winner["selected_confidence"]),
        "selected_top_k": selected_top_k,
        "selected_competition_mAP": float(winner["competition_mAP"]),
        "dataset_fingerprint": manifest.get("dataset_fingerprint"),
        "split_fingerprint": manifest.get("split_fingerprint"),
    }

    selection_rows.append({
        "model": model_name,
        "stage": stage,
        "experiment": experiment_id,
        "competition_mAP": float(winner["competition_mAP"]),
        "confidence": float(winner["selected_confidence"]),
        "top_k": selected_top_k,
        "checkpoint": source_checkpoint_path.name,
    })

MODEL_SELECTION_TABLE = pd.DataFrame(selection_rows).sort_values(
    ["competition_mAP", "model"], ascending=[False, True]
).reset_index(drop=True)

recommended = domain_summary["recommended"]
RECOMMENDED_MODEL = str(recommended["model"])
RECOMMENDED_COMPETITION_MAP = float(recommended["competition_mAP"])

if RECOMMENDED_MODEL not in MODEL_RUN_CONFIGS:
    raise RuntimeError(f"Recommended model is not an architecture winner: {RECOMMENDED_MODEL}")

display(MODEL_SELECTION_TABLE)
print(
    f"KAGGLE-DOMAIN RECOMMENDED: {RECOMMENDED_MODEL} / "
    f"{COMPETITION_METRIC}={RECOMMENDED_COMPETITION_MAP:.6f}"
)

selection_contract = {
    "pipeline_version": PIPELINE_VERSION,
    "created_at": datetime.now().isoformat(),
    "selection_source": str(summary_path),
    "selection_mode": "Kaggle-domain base Val 56, all 9 checkpoints",
    "models_to_submit": list(MODEL_RUN_CONFIGS),
    "recommended_model": RECOMMENDED_MODEL,
    "recommended_competition_mAP": RECOMMENDED_COMPETITION_MAP,
    "ranking": MODEL_SELECTION_TABLE.to_dict("records"),
    "models": MODEL_RUN_CONFIGS,
    "image_size": IMAGE_SIZE,
    "raw_prediction_confidence": RAW_PREDICTION_CONFIDENCE,
    "nms_iou_threshold": NMS_IOU_THRESHOLD,
    "max_detections": MAX_DETECTIONS,
    "split_fingerprint": domain_summary["split_fingerprint"],
    "integrated_dataset_tar": str(INTEGRATED_DATASET_TAR),
    "kaggle_test_images": EXPECTED_KAGGLE_TEST_IMAGES,
}

selection_signature = stable_json_hash(selection_contract)[:16]
FINAL_CONFIG_PATH = FINAL_CONFIG_DIR / f"integrated_submission_config_{selection_signature}.json"
write_json_atomic(FINAL_CONFIG_PATH, selection_contract)

print("Selection summary:", summary_path)
print("Submission contract:", FINAL_CONFIG_PATH)
show_stage_progress(5, "Load Kaggle-domain all-9 evaluation summary", "DONE")


[Pipeline 05/14 |  36%] START: Load Kaggle-domain all-9 evaluation summary
Reused verified local file: /content/baby_kangaroo_kaggle_submission_v30/model_input/kaggle_domain_eval_all9_cc5d16d3fe042c52_20260819_044743_summary.json
Reused verified local file: /content/baby_kangaroo_kaggle_submission_v30/model_input/YOLO11s_yolo11s_tune_b_45180ad6ea879a1b.json
Reused verified local file: /content/baby_kangaroo_kaggle_submission_v30/model_input/YOLO11s_b5153de50bdf_yolo11s_tune_b_45180ad6ea879a1b_best.pt
Reused verified local file: /content/baby_kangaroo_kaggle_submission_v30/model_input/kaggle_domain_eval_all9_cc5d16d3fe042c52_20260819_044743_summary.json
Reused verified local file: /content/baby_kangaroo_kaggle_submission_v30/model_input/YOLO11m_yolo11m_baseline_21cc7ae361705449.json
Reused verified local file: /content/baby_kangaroo_kaggle_submission_v30/model_input/YOLO11m_2139588cb913_yolo11m_baseline_21cc7ae361705449_best.pt
Reused verified local file: /content/baby_kangaroo_kaggle_s

,model,stage,experiment,competition_mAP,confidence,top_k,checkpoint
0,YOLO12m,tune_a,yolo12m_tune_a_402b815f89779a65,0.988119,0.001,4,yolo12m_tune_a_402b815f89779a65_best.pt
1,YOLO11m,baseline,yolo11m_baseline_21cc7ae361705449,0.987437,0.001,4,yolo11m_baseline_21cc7ae361705449_best.pt
2,YOLO11s,tune_b,yolo11s_tune_b_45180ad6ea879a1b,0.982130,0.001,4,yolo11s_tune_b_45180ad6ea879a1b_best.pt


KAGGLE-DOMAIN RECOMMENDED: YOLO12m / mAP@[0.75:0.95]=0.988119
Selection summary: /content/drive/MyDrive/baby_kangaroo/week2/공통파이프라인/reports/kaggle_domain_eval_all9_cc5d16d3fe042c52_20260819_044743_summary.json
Submission contract: /content/baby_kangaroo_kaggle_submission_v30/results/final_config/integrated_submission_config_605a510f5e79d59d.json
[Pipeline 05/14 |  36%] DONE: Load Kaggle-domain all-9 evaluation summary


# 3. Kaggle 입력 데이터 확인

`IntegratedDataset.tar`의 `inference_images/`를 제출용 테스트 이미지로 사용한다. 파일명에서 `image_id`를 만들고, 파일명·stem·변환된 ID의 중복 여부를 확인한다.

In [21]:
show_stage_progress(6, "Stage IntegratedDataset.tar inference_images", "START")

def collect_test_images(image_dir):
    """Kaggle Test 전체 이미지를 수집하고 파일명 충돌을 검사한다."""
    return validate_test_image_list(
        list_supported_images(image_dir),
        str(image_dir),
        expected_count=(
            SMOKE_IMAGE_LIMIT if RUN_MODE == "smoke"
            else EXPECTED_KAGGLE_TEST_IMAGES
        ),
    )


def parse_competition_image_id(file_name):
    """파일명의 숫자를 대회 image_id 정수로 변환하되 애매한 이름은 중단한다."""
    stem = Path(file_name).stem
    if stem.isdigit():
        image_id = int(stem)
    else:
        number_groups = re.findall(r"\d+", stem)
        if len(number_groups) != 1:
            raise ValueError(
                "The test filename must contain exactly one unambiguous numeric image ID: "
                f"{file_name!r}"
            )
        image_id = int(number_groups[0])
    if image_id < 1:
        raise ValueError(f"Kaggle image_id must be positive: {file_name!r}")
    return image_id


def build_test_image_id_mapping(image_paths):
    """전체 파일명과 대회 image_id 사이의 일대일 대응표를 만든다."""
    mapping_df = pd.DataFrame({
        "file_name": [path.name for path in image_paths],
        "image_id": [parse_competition_image_id(path.name) for path in image_paths],
    })
    if mapping_df["file_name"].duplicated().any():
        raise RuntimeError("Duplicate test filenames were found in the image ID mapping.")
    if mapping_df["image_id"].duplicated().any():
        duplicates = mapping_df.loc[
            mapping_df["image_id"].duplicated(keep=False),
            ["file_name", "image_id"],
        ]
        raise RuntimeError(
            "Multiple test files map to the same Kaggle image_id: "
            f"{duplicates.head(10).to_dict('records')}"
        )
    return mapping_df.sort_values("image_id").reset_index(drop=True)


KAGGLE_TEST_IMAGE_DIR, input_staging_mode, input_staging_manifest_df = (
    stage_kaggle_inputs_to_local()
)
LOCAL_STAGING_MANIFEST_PATH = REPORT_DIR / "local_input_staging_manifest.csv"
input_staging_manifest_df.to_csv(
    LOCAL_STAGING_MANIFEST_PATH,
    index=False,
    encoding="utf-8-sig",
)

all_test_image_paths = collect_test_images(KAGGLE_TEST_IMAGE_DIR)
test_image_id_mapping_df = build_test_image_id_mapping(all_test_image_paths)
TEST_IMAGE_ID_MAPPING_PATH = REPORT_DIR / "test_image_id_mapping.csv"
test_image_id_mapping_df.to_csv(
    TEST_IMAGE_ID_MAPPING_PATH,
    index=False,
    encoding="utf-8-sig",
)

active_test_image_paths = (
    all_test_image_paths[:SMOKE_IMAGE_LIMIT]
    if RUN_MODE == "smoke"
    else all_test_image_paths
)

print(f"Input staging mode: {input_staging_mode}")
print(f"Local Kaggle test root: {KAGGLE_TEST_IMAGE_DIR}")
print(f"IntegratedDataset TAR contains: {EXPECTED_KAGGLE_TEST_IMAGES} inference images")
print(f"Images staged this run: {len(all_test_image_paths)}")
print(f"Unique Kaggle image IDs: {test_image_id_mapping_df['image_id'].nunique()}")
print(f"Images used in this run: {len(active_test_image_paths)}")
print(f"Image ID mapping: {TEST_IMAGE_ID_MAPPING_PATH}")
display(test_image_id_mapping_df.head())

show_stage_progress(6, "Stage IntegratedDataset.tar inference_images", "DONE")

[Pipeline 06/14 |  43%] START: Stage IntegratedDataset.tar inference_images


Extract IntegratedDataset inference_images:   0%|          | 0/842 [00:00<?, ?image/s]

IntegratedDataset.tar inference_images extracted.
Input staging mode: integrated_dataset_tar_selective_inference_extract
Local Kaggle test root: /content/baby_kangaroo_kaggle_submission_v30/input/inference_images
IntegratedDataset TAR contains: 842 inference images
Images staged this run: 842
Unique Kaggle image IDs: 842
Images used in this run: 842
Image ID mapping: /content/baby_kangaroo_kaggle_submission_v30/results/reports/test_image_id_mapping.csv


,file_name,image_id
0,1.png,1
1,3.png,3
2,4.png,4
3,5.png,5
4,8.png,8


[Pipeline 06/14 |  43%] DONE: Stage IntegratedDataset.tar inference_images


In [22]:
# V2: sample_submission.csv가 있으면 공식 schema와 test image_id 집합을 직접 대조한다.
def locate_sample_submission():
    candidates = [
        Path(SAMPLE_SUBMISSION_PATH_OVERRIDE) if SAMPLE_SUBMISSION_PATH_OVERRIDE else None,
        LOCAL_INPUT_ROOT / "sample_submission.csv",
        Path("/content/sample_submission.csv"),
        SHARED_ROOT / "sample_submission.csv",
    ]
    return next((path for path in candidates if path is not None and path.is_file()), None)

SAMPLE_SUBMISSION_PATH = locate_sample_submission()
if SAMPLE_SUBMISSION_PATH is None:
    print("sample_submission.csv not found: object-row schema remains code-validated; image_id audit uses test filenames.")
else:
    sample_submission_df = pd.read_csv(SAMPLE_SUBMISSION_PATH)
    if sample_submission_df.columns.tolist() != KAGGLE_SUBMISSION_COLUMNS:
        raise RuntimeError(
            f"sample_submission schema mismatch: {sample_submission_df.columns.tolist()}"
        )
    sample_ids = set(pd.to_numeric(sample_submission_df["image_id"], errors="coerce").dropna().astype(int))
    mapped_ids = set(test_image_id_mapping_df["image_id"].astype(int))
    # sample이 모든 test image를 row로 표현하는 형식일 때만 집합 일치를 강제한다.
    if sample_ids and len(sample_ids) >= len(mapped_ids) and sample_ids != mapped_ids:
        raise RuntimeError(
            f"sample_submission image_id mismatch: missing={sorted(mapped_ids-sample_ids)[:10]}, "
            f"extra={sorted(sample_ids-mapped_ids)[:10]}"
        )
    print(f"sample_submission audit PASS: {SAMPLE_SUBMISSION_PATH}")


sample_submission.csv not found: object-row schema remains code-validated; image_id audit uses test filenames.


# 4. 테스트 이미지 전처리

학습 데이터와 같은 순서로 EXIF·ICC를 정규화하고 bilateral filter, 제한된 gray-world white balance, LAB L 채널 CLAHE, 960×960 letterbox를 적용한다. 테스트 이미지에는 증강을 적용하지 않는다.

원본 크기, scale, padding은 manifest에 저장해 추론 후 BBox를 원본 좌표로 복원할 때 사용한다.

In [23]:
show_stage_progress(7, "Define deterministic v6.0 preprocessing", "START")

# v6.0 학습 데이터와 동일한 결정적 전처리 설정이다.
PREPROCESS_TARGET_SIZE = 960
PREPROCESS_DENOISE_DIAMETER = 5
PREPROCESS_DENOISE_SIGMA_COLOR = 20.0
PREPROCESS_DENOISE_SIGMA_SPACE = 20.0
PREPROCESS_WHITE_BALANCE_GAIN_LIMIT = 0.05
PREPROCESS_CLAHE_CLIP_LIMIT = 1.5
PREPROCESS_CLAHE_GRID = (8, 8)

PREPROCESSING_CONFIG = {
    "order": [
        "EXIF orientation normalization",
        "ICC-aware sRGB conversion",
        "edge-preserving bilateral denoising",
        "clipped gray-world white balance",
        "LAB luminance CLAHE",
        "aspect-ratio-preserving letterbox resize",
    ],
    "target_size": [PREPROCESS_TARGET_SIZE, PREPROCESS_TARGET_SIZE],
    "denoise": {
        "method": "cv2.bilateralFilter",
        "diameter": PREPROCESS_DENOISE_DIAMETER,
        "sigma_color": PREPROCESS_DENOISE_SIGMA_COLOR,
        "sigma_space": PREPROCESS_DENOISE_SIGMA_SPACE,
    },
    "color_correction": {
        "white_balance": "gray_world_with_clipped_channel_gains",
        "gain_limit": PREPROCESS_WHITE_BALANCE_GAIN_LIMIT,
        "luminance_equalization": "CLAHE_on_LAB_L_only",
        "clahe_clip_limit": PREPROCESS_CLAHE_CLIP_LIMIT,
        "clahe_grid": list(PREPROCESS_CLAHE_GRID),
    },
    "letterbox_padding": "median RGB of the corrected image border",
    "augmentation": "disabled_for_kaggle_inference",
}

if PREPROCESS_TARGET_SIZE != IMAGE_SIZE:
    raise RuntimeError("Kaggle preprocessing size must match the model input size.")


# preprocessing_config.json과 현재 전처리 설정이 일치하는지 확인한다.
DRIVE_PREPROCESSING_CONFIG = json.loads(
    PREPROCESS_CONFIG_PATH.read_text(encoding="utf-8")
)

EXPECTED_PREPROCESSING_ORDER = [
    "EXIF orientation normalization",
    "ICC-aware sRGB conversion",
    "edge-preserving bilateral denoising",
    "clipped gray-world white balance",
    "LAB luminance CLAHE",
    "aspect-ratio-preserving letterbox resize",
]

if DRIVE_PREPROCESSING_CONFIG.get("order") != EXPECTED_PREPROCESSING_ORDER:
    raise RuntimeError(
        "Latest V6.0.5 preprocessing order changed. "
        "Do not run Kaggle inference with this notebook."
    )

if list(map(int, DRIVE_PREPROCESSING_CONFIG.get("target_size", []))) != [
    PREPROCESS_TARGET_SIZE, PREPROCESS_TARGET_SIZE
]:
    raise RuntimeError("Preprocessing target size mismatch.")

drive_denoise = DRIVE_PREPROCESSING_CONFIG.get("denoise", {})
if (
    drive_denoise.get("method") != "cv2.bilateralFilter"
    or int(drive_denoise.get("diameter", -1)) != PREPROCESS_DENOISE_DIAMETER
    or float(drive_denoise.get("sigma_color", -1)) != PREPROCESS_DENOISE_SIGMA_COLOR
    or float(drive_denoise.get("sigma_space", -1)) != PREPROCESS_DENOISE_SIGMA_SPACE
):
    raise RuntimeError("V6.0.5 denoise contract mismatch.")

drive_color = DRIVE_PREPROCESSING_CONFIG.get("color_correction", {})
if (
    drive_color.get("white_balance") != "gray_world_with_clipped_channel_gains"
    or float(drive_color.get("gain_limit", -1)) != PREPROCESS_WHITE_BALANCE_GAIN_LIMIT
    or drive_color.get("luminance_equalization") != "CLAHE_on_LAB_L_only"
    or float(drive_color.get("clahe_clip_limit", -1)) != PREPROCESS_CLAHE_CLIP_LIMIT
    or list(map(int, drive_color.get("clahe_grid", []))) != list(PREPROCESS_CLAHE_GRID)
):
    raise RuntimeError("V6.0.5 color-correction contract mismatch.")

if DRIVE_PREPROCESSING_CONFIG.get("letterbox_padding") != (
    "median RGB of the corrected image border"
):
    raise RuntimeError("V6.0.5 letterbox padding contract mismatch.")

print("Latest V6.0.5 preprocessing_config.json contract: PASS")
print("image_to_srgb: ICC -> sRGB before alpha composite")


class ColorProfileError(ValueError):
    """V6.0.5 전처리의 ICC 변환 실패를 구분하기 위한 전용 예외."""


def image_to_srgb(image):
    """ICC 프로필을 sRGB로 변환한 뒤 알파 채널을 합성한다."""
    normalized = ImageOps.exif_transpose(image)
    icc_profile = normalized.info.get("icc_profile")
    has_alpha = "A" in normalized.getbands() or "transparency" in normalized.info

    # 흰 배경 합성보다 ICC -> sRGB 변환을 먼저 수행한다.
    if icc_profile:
        try:
            source_profile = ImageCms.ImageCmsProfile(BytesIO(icc_profile))

            # 이미 sRGB면 결과를 바꾸지 않는 불필요한 변환을 건너뛴다.
            profile_description = ""
            try:
                profile_description = str(
                    ImageCms.getProfileDescription(source_profile) or ""
                ).strip().lower()
            except Exception:
                profile_description = ""

            already_srgb = "srgb" in profile_description.replace(" ", "")
            if not already_srgb:
                working_mode = "RGBA" if has_alpha else "RGB"
                target_profile = ImageCms.createProfile("sRGB")
                normalized = ImageCms.profileToProfile(
                    normalized.convert(working_mode),
                    source_profile,
                    target_profile,
                    outputMode=working_mode,
                    renderingIntent=0,
                )
        except Exception as error:
            raise ColorProfileError(
                f"ICC profile conversion failed: {error}"
            ) from error

    # 색공간을 sRGB로 맞춘 뒤 알파를 흰색 배경에 합성한다.
    if has_alpha:
        rgba = normalized.convert("RGBA")
        background = Image.new("RGB", rgba.size, "white")
        background.paste(rgba, mask=rgba.getchannel("A"))
        normalized = background
    else:
        normalized = normalized.convert("RGB")

    return normalized

def border_median_rgb(rgb_array, border_fraction=0.04):
    """letterbox 여백에 사용할 이미지 테두리의 중앙 RGB 값을 계산한다."""
    height, width = rgb_array.shape[:2]
    thickness = max(1, int(round(min(height, width) * border_fraction)))
    border_pixels = np.concatenate([
        rgb_array[:thickness].reshape(-1, 3),
        rgb_array[-thickness:].reshape(-1, 3),
        rgb_array[:, :thickness].reshape(-1, 3),
        rgb_array[:, -thickness:].reshape(-1, 3),
    ], axis=0)
    return np.median(border_pixels, axis=0).round().clip(0, 255).astype(np.uint8)


def clipped_gray_world_white_balance(rgb_array):
    """채널별 gain을 제한해 과보정 없이 gray-world 화이트밸런스를 적용한다."""
    work = rgb_array.astype(np.float32)
    channel_means = work.reshape(-1, 3).mean(axis=0)
    gray_mean = float(channel_means.mean())
    raw_gains = gray_mean / np.maximum(channel_means, 1.0)
    low = 1.0 - PREPROCESS_WHITE_BALANCE_GAIN_LIMIT
    high = 1.0 + PREPROCESS_WHITE_BALANCE_GAIN_LIMIT
    gains = np.clip(raw_gains, low, high)
    balanced = np.clip(work * gains.reshape(1, 1, 3), 0, 255).astype(np.uint8)
    return balanced, gains


def clahe_luminance_only(rgb_array):
    """색상 채널은 유지하고 LAB의 밝기 채널에만 CLAHE를 적용한다."""
    lab = cv2.cvtColor(rgb_array, cv2.COLOR_RGB2LAB)
    l_channel, a_channel, b_channel = cv2.split(lab)
    clahe = cv2.createCLAHE(
        clipLimit=PREPROCESS_CLAHE_CLIP_LIMIT,
        tileGridSize=PREPROCESS_CLAHE_GRID,
    )
    enhanced_l = clahe.apply(l_channel)
    return cv2.cvtColor(
        cv2.merge((enhanced_l, a_channel, b_channel)),
        cv2.COLOR_LAB2RGB,
    )


def save_rgb_image(rgb_array, destination):
    """원본 확장자에 맞춰 전처리 RGB 이미지를 저장한다."""
    destination = Path(destination)
    image = Image.fromarray(rgb_array, mode="RGB")
    suffix = destination.suffix.lower()
    if suffix in {".jpg", ".jpeg"}:
        image.save(destination, format="JPEG", quality=95, subsampling=0)
    elif suffix == ".png":
        image.save(destination, format="PNG", compress_level=3)
    else:
        image.save(destination)


def preprocess_kaggle_image(source, destination):
    """학습 데이터와 같은 결정적 전처리를 적용하고 역변환 정보를 반환한다."""
    source = Path(source)
    destination = Path(destination)
    with Image.open(source) as raw_image:
        rgb = np.asarray(image_to_srgb(raw_image), dtype=np.uint8)

    source_height, source_width = rgb.shape[:2]
    if source_width <= 0 or source_height <= 0:
        raise ValueError(f"Invalid image size: {source}")

    denoised = cv2.bilateralFilter(
        rgb,
        d=PREPROCESS_DENOISE_DIAMETER,
        sigmaColor=PREPROCESS_DENOISE_SIGMA_COLOR,
        sigmaSpace=PREPROCESS_DENOISE_SIGMA_SPACE,
    )
    white_balanced, wb_gains = clipped_gray_world_white_balance(denoised)
    corrected = clahe_luminance_only(white_balanced)

    nominal_scale = min(
        PREPROCESS_TARGET_SIZE / source_width,
        PREPROCESS_TARGET_SIZE / source_height,
    )
    resized_width = max(1, int(round(source_width * nominal_scale)))
    resized_height = max(1, int(round(source_height * nominal_scale)))
    interpolation = cv2.INTER_AREA if nominal_scale < 1.0 else cv2.INTER_LANCZOS4
    resized = cv2.resize(corrected, (resized_width, resized_height), interpolation=interpolation)

    pad_left = (PREPROCESS_TARGET_SIZE - resized_width) // 2
    pad_top = (PREPROCESS_TARGET_SIZE - resized_height) // 2
    padding_rgb = border_median_rgb(corrected)
    canvas = np.empty((PREPROCESS_TARGET_SIZE, PREPROCESS_TARGET_SIZE, 3), dtype=np.uint8)
    canvas[...] = padding_rgb
    canvas[pad_top:pad_top + resized_height, pad_left:pad_left + resized_width] = resized
    save_rgb_image(canvas, destination)

    # 저장된 전처리 결과 파일의 SHA256을 계산한다(캐시 무결성 검증용).
    processed_sha256 = sha256_file(destination)

    return {
        "file_name": source.name,
        "source_width": int(source_width),
        "source_height": int(source_height),
        "output_width": PREPROCESS_TARGET_SIZE,
        "output_height": PREPROCESS_TARGET_SIZE,
        "resized_width": int(resized_width),
        "resized_height": int(resized_height),
        "scale_x": float(resized_width / source_width),
        "scale_y": float(resized_height / source_height),
        "pad_left": int(pad_left),
        "pad_top": int(pad_top),
        "padding_r": int(padding_rgb[0]),
        "padding_g": int(padding_rgb[1]),
        "padding_b": int(padding_rgb[2]),
        "wb_gain_r": float(wb_gains[0]),
        "wb_gain_g": float(wb_gains[1]),
        "wb_gain_b": float(wb_gains[2]),
        "processed_sha256": processed_sha256,  # 캐시 재사용 검증을 위한 SHA256 필드
    }

show_stage_progress(7, "Define deterministic v6.0 preprocessing", "DONE")

[Pipeline 07/14 |  50%] START: Define deterministic v6.0 preprocessing
Latest V6.0.5 preprocessing_config.json contract: PASS
image_to_srgb implementation: FIX17/18 (ICC->sRGB BEFORE alpha composite)
[Pipeline 07/14 |  50%] DONE: Define deterministic v6.0 preprocessing


In [24]:
show_stage_progress(8, "Preprocess and cache active test images", "START")

# Kaggle 입력 해시·캐시 확인·전처리에 걸린 시간을 함께 측정한다.
preprocessing_started_at = datetime.now()
preprocessing_started_perf = time.perf_counter()

# 활성 이미지의 파일 지문을 이용해 동일한 전처리 결과만 재사용한다.
active_inventory = []
for source_path in tqdm(active_test_image_paths, desc="Hash Kaggle test images", unit="image"):
    active_inventory.append({
        "file_name": source_path.name,
        "source_path": str(source_path),
        "source_sha256": sha256_file(source_path),
        "source_size_bytes": source_path.stat().st_size,
    })

preprocessing_contract = {
    "run_mode": RUN_MODE,
    "config": PREPROCESSING_CONFIG,
    "images": active_inventory,
}
preprocessing_signature = stable_json_hash(preprocessing_contract)[:16]
ACTIVE_PREPROCESSED_DIR = PREPROCESSED_ROOT / preprocessing_signature / "images"
PREPROCESSING_MANIFEST_PATH = (
    PREPROCESSED_ROOT / preprocessing_signature / "preprocessing_manifest.csv"
)
ACTIVE_PREPROCESSED_DIR.mkdir(parents=True, exist_ok=True)

reuse_preprocessing = False
if PREPROCESSING_MANIFEST_PATH.is_file():
    cached_manifest = pd.read_csv(PREPROCESSING_MANIFEST_PATH)

    # 파일 존재 여부만이 아니라 manifest에 기록된 processed_sha256과 실제 파일의
    # 해시값이 일치하는지까지 확인해야 캐시가 정말 유효한지 알 수 있다.
    if "processed_sha256" in cached_manifest.columns:
        reuse_preprocessing = (
            len(cached_manifest) == len(active_test_image_paths)
            and cached_manifest["file_name"].is_unique
            and all(
                (ACTIVE_PREPROCESSED_DIR / row["file_name"]).is_file() and
                sha256_file(ACTIVE_PREPROCESSED_DIR / row["file_name"]) == row["processed_sha256"]
                for _, row in cached_manifest.iterrows()
            )
        )
    else:
        reuse_preprocessing = False

if reuse_preprocessing:
    preprocessing_manifest_df = cached_manifest
    print(f"Reused verified Kaggle preprocessing cache: {PREPROCESSING_MANIFEST_PATH}")
else:
    inventory_by_name = {row["file_name"]: row for row in active_inventory}
    transform_rows = []
    for source_path in tqdm(
        active_test_image_paths,
        desc="Preprocess Kaggle test images",
        unit="image",
    ):
        destination_path = ACTIVE_PREPROCESSED_DIR / source_path.name
        transform = preprocess_kaggle_image(source_path, destination_path)
        transform_rows.append({
            **transform,
            "source_path": str(source_path),
            "processed_path": str(destination_path),
            "source_sha256": inventory_by_name[source_path.name]["source_sha256"],
            "processed_sha256": sha256_file(destination_path),
            "preprocessing_signature": preprocessing_signature,
        })
    preprocessing_manifest_df = pd.DataFrame(transform_rows).sort_values("file_name")
    preprocessing_manifest_df.to_csv(
        PREPROCESSING_MANIFEST_PATH,
        index=False,
        encoding="utf-8-sig",
    )

if len(preprocessing_manifest_df) != len(active_test_image_paths):
    raise RuntimeError("Kaggle preprocessing manifest count mismatch.")
if not (
    (preprocessing_manifest_df["output_width"] == IMAGE_SIZE).all()
    and (preprocessing_manifest_df["output_height"] == IMAGE_SIZE).all()
):
    raise RuntimeError("A preprocessed Kaggle image is not 960x960.")

print(f"Preprocessed images: {len(preprocessing_manifest_df)}")
print(f"Preprocessing manifest: {PREPROCESSING_MANIFEST_PATH}")

preprocessing_seconds = time.perf_counter() - preprocessing_started_perf
preprocessing_finished_at = datetime.now()
print(f"Preprocessing seconds: {preprocessing_seconds:.2f}")

show_stage_progress(8, "Preprocess and cache active test images", "DONE")

[Pipeline 08/14 |  57%] START: Preprocess and cache active test images


Hash Kaggle test images:   0%|          | 0/842 [00:00<?, ?image/s]

Preprocess Kaggle test images:   0%|          | 0/842 [00:00<?, ?image/s]

/tmp/ipykernel_742/3826341700.py:191: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  image = Image.fromarray(rgb_array, mode="RGB")


Preprocessed images: 842
Preprocessing manifest: /content/baby_kangaroo_kaggle_submission_v30/results/preprocessed_test/14ebf6bb7189a8b6/preprocessing_manifest.csv
Preprocessing seconds: 246.65
[Pipeline 08/14 |  57%] DONE: Preprocess and cache active test images


# 5. 체크포인트와 클래스 매핑 확인

세 모델의 클래스 매핑을 확인한다. GPU 메모리를 줄이기 위해 모델은 한 번에 하나씩 로드하고, 해당 모델의 추론이 끝나면 메모리에서 해제한다.

In [25]:
show_stage_progress(9, "Validate mappings and define memory-safe model loaders", "START")


def load_class_mapping(model_name):
    """모델별 class_mapping.csv를 검증해 원본 category ID 복원표를 만든다."""
    source_mapping_path = YOLO_BUNDLE_DIRS[model_name] / "class_mapping.csv"
    model_label_column = "yolo_class_id"
    class_name_candidates = ["normalized_class_name", "class_name", "original_class_name"]

    if not source_mapping_path.is_file():
        raise FileNotFoundError(f"Class mapping not found: {source_mapping_path}")
    mapping_path = LOCAL_MODEL_INPUT_DIR / f"{model_name}_class_mapping.csv"
    copy_file_verified(
        source_mapping_path,
        mapping_path,
        expected_sha256=sha256_file(source_mapping_path),
    )
    mapping_df = pd.read_csv(mapping_path)
    required = {model_label_column, "original_category_id"}
    missing = required - set(mapping_df.columns)
    if missing:
        raise ValueError(f"Class mapping columns missing: {sorted(missing)}")
    if len(mapping_df) != EXPECTED_CLASSES:
        raise RuntimeError(f"Expected {EXPECTED_CLASSES} classes, found {len(mapping_df)}.")

    mapping_df[model_label_column] = pd.to_numeric(
        mapping_df[model_label_column], errors="raise"
    ).astype(int)
    mapping_df["original_category_id"] = pd.to_numeric(
        mapping_df["original_category_id"], errors="raise"
    ).astype(int)
    if mapping_df[model_label_column].duplicated().any():
        raise RuntimeError(f"Duplicate {model_name} model labels were found.")
    if sorted(mapping_df[model_label_column].tolist()) != list(range(EXPECTED_CLASSES)):
        raise RuntimeError(
            f"{model_name} YOLO class IDs must be exactly 0..{EXPECTED_CLASSES - 1}."
        )
    if mapping_df["original_category_id"].duplicated().any():
        raise RuntimeError(f"Duplicate {model_name} original category IDs were found.")

    class_name_column = next(
        (column for column in class_name_candidates if column in mapping_df.columns),
        None,
    )
    model_to_original = dict(zip(
        mapping_df[model_label_column],
        mapping_df["original_category_id"],
    ))
    kaggle_official_model_to_original = {
        int(k): int(v)
        for k, v in model_to_original.items()
        if int(k) in KAGGLE_OFFICIAL_YOLO_CLASS_IDS
    }
    if sorted(kaggle_official_model_to_original) != list(KAGGLE_OFFICIAL_YOLO_CLASS_IDS):
        raise RuntimeError(
            f"{model_name} official Kaggle class mapping must contain YOLO IDs 0.."
            f"{KAGGLE_OFFICIAL_CLASS_COUNT - 1}."
        )
    original_to_name = {
        int(row["original_category_id"]): (
            str(row[class_name_column]) if class_name_column else str(row["original_category_id"])
        )
        for _, row in mapping_df.iterrows()
    }
    return {
        "mapping_path": mapping_path,
        "mapping_df": mapping_df,
        "model_label_column": model_label_column,
        "model_to_original": model_to_original,
        "kaggle_official_model_to_original": kaggle_official_model_to_original,
        "original_to_name": original_to_name,
        "allowed_original_category_ids": set(model_to_original.values()),
        "kaggle_official_original_category_ids": set(kaggle_official_model_to_original.values()),
    }


def load_inference_bundle(model_name):
    """지정한 모델 하나만 메모리에 불러와 추론 context를 반환한다."""
    config = MODEL_RUN_CONFIGS[model_name]
    mapping = CLASS_MAPPING_CONTRACTS[model_name]
    checkpoint_path = Path(config["local_checkpoint_path"])

    model = YOLO(str(checkpoint_path))

    return {
        "model_name": model_name,
        "model": model,
        "config": config,
        "mapping": mapping,
    }


def release_inference_bundle(bundle):
    """완료된 모델을 CPU로 내리고 참조·CUDA 캐시를 정리한다."""
    model = bundle.pop("model", None) if bundle is not None else None
    if model is not None:
        try:
            if hasattr(model, "predictor"):
                model.predictor = None
            if hasattr(model, "model") and isinstance(model.model, torch.nn.Module):
                model.model.to("cpu")
            elif isinstance(model, torch.nn.Module):
                model.to("cpu")
        except Exception:
            pass
        del model
        gc.collect()
        if torch.cuda.is_available() and torch.version.cuda is not None:
            try:
                torch.cuda.empty_cache()
                torch.cuda.ipc_collect()
            except (RuntimeError, AssertionError):
                pass


def current_gpu_memory_gb():
    """현재 PyTorch CUDA 메모리 사용량을 GB로 반환한다."""
    if not torch.cuda.is_available():
        return 0.0
    return float(torch.cuda.memory_allocated() / (1024 ** 3))


CLASS_MAPPING_CONTRACTS = {}
mapping_rows = []
for model_name in tqdm(MODELS_TO_SUBMIT, desc="Validate class mappings", unit="model"):
    mapping_contract = load_class_mapping(model_name)
    CLASS_MAPPING_CONTRACTS[model_name] = mapping_contract
    mapping_rows.append({
        "model": model_name,
        "classes": len(mapping_contract["mapping_df"]),
        "original_category_ids": len(mapping_contract["allowed_original_category_ids"]),
        "mapping_path": str(mapping_contract["mapping_path"]),
    })

display(pd.DataFrame(mapping_rows))
show_stage_progress(9, "Validate mappings and define memory-safe model loaders", "DONE")

[Pipeline 09/14 |  64%] START: Validate mappings and define memory-safe model loaders


Validate class mappings:   0%|          | 0/3 [00:00<?, ?model/s]

Reused verified local file: /content/baby_kangaroo_kaggle_submission_v30/model_input/YOLO11s_class_mapping.csv
Reused verified local file: /content/baby_kangaroo_kaggle_submission_v30/model_input/YOLO11m_class_mapping.csv
Reused verified local file: /content/baby_kangaroo_kaggle_submission_v30/model_input/YOLO12m_class_mapping.csv


,model,classes,original_category_ids,mapping_path
0,YOLO11s,118,118,/content/baby_kangaroo_kaggle_submission_v30/m...
1,YOLO11m,118,118,/content/baby_kangaroo_kaggle_submission_v30/m...
2,YOLO12m,118,118,/content/baby_kangaroo_kaggle_submission_v30/m...


[Pipeline 09/14 |  64%] DONE: Validate mappings and define memory-safe model loaders


# 6. 테스트 추론과 좌표 복원

960×960 전처리 좌표에서 얻은 예측 BBox를 저장된 scale과 padding으로 원본 이미지 좌표에 복원한다. 모델별 Validation confidence와 체크포인트는 각각 독립적으로 적용한다.

In [26]:
show_stage_progress(10, "Define three-model inference functions", "START")

PREDICTION_COLUMNS = [
    "file_name",
    "category_id",
    "score",
    "x_min",
    "y_min",
    "x_max",
    "y_max",
    "x",
    "y",
    "width",
    "height",
    "rank",
    "model",
]


def inverse_letterbox_box(box_xyxy, transform):
    """960x960 xyxy BBox를 원본 이미지 좌표로 복원하고 범위 안으로 자른다."""
    x_min, y_min, x_max, y_max = map(float, box_xyxy)
    x_min = (x_min - float(transform["pad_left"])) / float(transform["scale_x"])
    x_max = (x_max - float(transform["pad_left"])) / float(transform["scale_x"])
    y_min = (y_min - float(transform["pad_top"])) / float(transform["scale_y"])
    y_max = (y_max - float(transform["pad_top"])) / float(transform["scale_y"])

    source_width = float(transform["source_width"])
    source_height = float(transform["source_height"])
    x_min = float(np.clip(x_min, 0.0, source_width))
    x_max = float(np.clip(x_max, 0.0, source_width))
    y_min = float(np.clip(y_min, 0.0, source_height))
    y_max = float(np.clip(y_max, 0.0, source_height))
    if not all(math.isfinite(value) for value in [x_min, y_min, x_max, y_max]):
        raise ValueError("A restored BBox contains NaN or infinity.")
    if x_max <= x_min or y_max <= y_min:
        return None
    return [x_min, y_min, x_max, y_max]


def normalize_image_predictions(
    file_name,
    boxes,
    labels,
    scores,
    transform,
    model_name,
    selected_confidence,
    selected_top_k,
    model_to_original,
):
    """한 모델의 예측을 원본 category ID와 원본 좌표의 공통 행으로 변환한다."""
    candidates = []
    for box, label, score in zip(boxes, labels, scores):
        score = float(score)
        if score < selected_confidence:
            continue
        model_label = int(label)

        # IMPORTANT: Kaggle evaluates only the original 56 competition classes.
        # Filter the expanded classes BEFORE sorting and Top-K so they cannot consume
        # one of the per-image Top-K slots.
        if model_label not in KAGGLE_OFFICIAL_YOLO_CLASS_IDS:
            continue

        if model_label not in model_to_original:
            raise KeyError(f"Unknown {model_name} label {model_label} in {file_name}.")
        restored = inverse_letterbox_box(box, transform)
        if restored is None:
            continue
        x_min, y_min, x_max, y_max = restored
        candidates.append({
            "file_name": file_name,
            "category_id": int(model_to_original[model_label]),
            "score": score,
            "x_min": x_min,
            "y_min": y_min,
            "x_max": x_max,
            "y_max": y_max,
            "x": x_min,
            "y": y_min,
            "width": x_max - x_min,
            "height": y_max - y_min,
            "model": model_name,
        })

    candidates.sort(key=lambda row: row["score"], reverse=True)
    selected = candidates if selected_top_k is None else candidates[:int(selected_top_k)]
    for rank, row in enumerate(selected, start=1):
        row["rank"] = rank
    return selected


def run_yolo_kaggle_inference(bundle, manifest_df):
    """YOLO를 디렉터리 stream·batch configurable로 실행해 목록 일괄 GPU 로딩을 막는다."""
    model_name = bundle["model_name"]
    model = bundle["model"]
    config = bundle["config"]
    mapping = bundle["mapping"]
    transform_by_name = manifest_df.set_index("file_name").to_dict("index")
    processed_paths = [Path(path) for path in manifest_df["processed_path"].tolist()]
    processed_directories = {path.resolve().parent for path in processed_paths}
    if len(processed_directories) != 1:
        raise RuntimeError(f"{model_name} preprocessed images must share one directory.")
    processed_directory = next(iter(processed_directories))

    expected_names = set(manifest_df["file_name"])
    actual_names = {path.name for path in list_supported_images(processed_directory)}
    if actual_names != expected_names:
        raise RuntimeError(
            f"{model_name} streaming directory does not match the preprocessing manifest. "
            f"missing={sorted(expected_names - actual_names)[:5]}, "
            f"extra={sorted(actual_names - expected_names)[:5]}"
        )

    # USE_TTA가 문자열 'False'일 때 bool('False') == True가 되어 옵션이 무력화(항상 켜짐)
    # 되는 것을 막기 위해 명시적으로 파싱한다. TTA가 켜지면 메모리 사용량이 급증하므로
    # 메모리 사용량을 줄이기 위해 batch size를 1로 설정한다.
    is_tta_enabled = (
        str(USE_TTA).strip().lower() in ["true", "1", "t", "y", "yes"]
        if isinstance(USE_TTA, str)
        else bool(USE_TTA)
    )

    predict_kwargs = {
        "source": str(processed_directory),
        "stream": True,
        "batch": 1 if is_tta_enabled else YOLO_INFERENCE_BATCH_SIZE,
        "imgsz": IMAGE_SIZE,
        "conf": RAW_PREDICTION_CONFIDENCE,
        "iou": NMS_IOU_THRESHOLD,
        "max_det": MAX_DETECTIONS,
        "classes": list(KAGGLE_OFFICIAL_YOLO_CLASS_IDS),
        "device": 0 if torch.cuda.is_available() else "cpu",
        "verbose": False,
        "augment": is_tta_enabled,
    }

    rows = []
    seen_names = set()
    ordered_expected_names = [path.name for path in processed_paths]
    results = model.predict(**predict_kwargs)
    for result_index, result in enumerate(tqdm(
        results,
        total=len(processed_paths),
        desc=f"{model_name} Kaggle inference",
        unit="image",
    )):
        returned_name = Path(str(getattr(result, "path", ""))).name
        file_name = returned_name
        if file_name not in transform_by_name:
            generic_name = bool(re.fullmatch(r"image\d+\.[A-Za-z0-9]+", file_name))
            if generic_name and result_index < len(ordered_expected_names):
                file_name = ordered_expected_names[result_index]
            else:
                raise KeyError(
                    f"{model_name} returned an unknown image name: {returned_name!r}"
                )
        if file_name in seen_names:
            raise RuntimeError(f"{model_name} returned a duplicate image result: {file_name}")
        seen_names.add(file_name)

        boxes = result.boxes
        if boxes is None or len(boxes) == 0:
            continue
        rows.extend(normalize_image_predictions(
            file_name,
            boxes.xyxy.detach().cpu().numpy(),
            boxes.cls.detach().cpu().numpy(),
            boxes.conf.detach().cpu().numpy(),
            transform_by_name[file_name],
            model_name,
            float(config["selected_confidence"]),
            config.get("selected_top_k"),
            mapping["kaggle_official_model_to_original"],
        ))

    if seen_names != expected_names:
        raise RuntimeError(
            f"{model_name} result count mismatch: "
            f"expected={len(expected_names)}, returned={len(seen_names)}"
        )
    return rows


show_stage_progress(10, "Define three-model inference functions", "DONE")

[Pipeline 10/14 |  71%] START: Define three-model inference functions
[Pipeline 10/14 |  71%] DONE: Define three-model inference functions


In [27]:
show_stage_progress(11, "Run YOLO11s, YOLO11m, and YOLO12m inference", "START")


def build_inference_manifest(raw_predictions_df, preprocessing_manifest):
    """한 모델의 이미지별 최종 검출 개수를 포함한 추론 manifest를 만든다."""
    prediction_counts = (
        raw_predictions_df.groupby("file_name").size().to_dict()
        if not raw_predictions_df.empty
        else {}
    )
    manifest_df = preprocessing_manifest[[
        "file_name", "source_path", "source_width", "source_height"
    ]].copy()
    manifest_df["prediction_count"] = manifest_df["file_name"].map(
        prediction_counts
    ).fillna(0).astype(int)
    return manifest_df


def validate_model_inference_output(
    model_name,
    raw_predictions_df,
    inference_manifest_df,
    preprocessing_manifest,
    allowed_category_ids,
):
    """캐시 또는 새 추론 결과가 현재 입력·클래스 계약과 맞는지 확인한다."""
    if raw_predictions_df.columns.tolist() != PREDICTION_COLUMNS:
        raise RuntimeError(f"{model_name} raw prediction columns do not match the contract.")
    expected_names = set(preprocessing_manifest["file_name"])
    actual_names = set(inference_manifest_df["file_name"])
    if len(inference_manifest_df) != len(preprocessing_manifest) or actual_names != expected_names:
        raise RuntimeError(f"{model_name} inference manifest image contract failed.")
    if inference_manifest_df["file_name"].duplicated().any():
        raise RuntimeError(f"{model_name} inference manifest contains duplicate files.")

    if MAX_PILLS_PER_IMAGE is not None:
        if (inference_manifest_df["prediction_count"] > MAX_PILLS_PER_IMAGE).any():
            raise ValueError(f"Image produced more than {MAX_PILLS_PER_IMAGE} pills")

    unknown_categories = set(
        raw_predictions_df.get("category_id", pd.Series(dtype=int)).astype(int)
    ) - set(map(int, allowed_category_ids))
    if unknown_categories:
        raise RuntimeError(f"{model_name} produced unknown category IDs: {unknown_categories}")
    if not raw_predictions_df.empty and set(raw_predictions_df["file_name"]) - expected_names:
        raise RuntimeError(f"{model_name} predictions contain unknown filenames.")


def execute_one_model_inference(model_name, preprocessing_manifest):
    """모델 하나를 추론하거나 검증된 로컬 캐시를 재사용한 뒤 결과만 반환한다."""
    config = MODEL_RUN_CONFIGS[model_name]
    mapping = CLASS_MAPPING_CONTRACTS[model_name]

    # selected_top_k/use_tta/max_detections/runtime_versions도 서명에 포함한다 -
    # 이 값들이 바뀌면 이전 캐시를 그대로 재사용하면 안 되기 때문이다.
    inference_contract = {
        "pipeline_version": PIPELINE_VERSION,
        "run_mode": RUN_MODE,
        "model_name": model_name,
        "checkpoint_sha256": config["checkpoint_sha256"],
        "selected_confidence": config["selected_confidence"],
        "selected_top_k": config.get("selected_top_k"),
        "use_tta": USE_TTA,
        "max_detections": MAX_DETECTIONS,
        "kaggle_official_class_count": KAGGLE_OFFICIAL_CLASS_COUNT,
        "kaggle_official_yolo_class_ids": list(KAGGLE_OFFICIAL_YOLO_CLASS_IDS),
        "class_filter_applied_before_top_k": True,
        "preprocessing_signature": preprocessing_signature,
        "images": preprocessing_manifest["file_name"].tolist(),
        "image_size": IMAGE_SIZE,
        "raw_confidence": RAW_PREDICTION_CONFIDENCE,
        "nms_iou": NMS_IOU_THRESHOLD,
        "max_pills_per_image": MAX_PILLS_PER_IMAGE,
        "inference_batch_size": YOLO_INFERENCE_BATCH_SIZE,
        "runtime_versions": RUNTIME_VERSIONS,
    }
    inference_signature = stable_json_hash(inference_contract)[:16]
    raw_path = PREDICTION_DIR / f"raw_predictions_{model_name.lower()}_{inference_signature}.csv"
    manifest_path = PREDICTION_DIR / f"inference_manifest_{model_name.lower()}_{inference_signature}.csv"
    cache_marker_path = PREDICTION_DIR / f"inference_cache_{model_name.lower()}_{inference_signature}.json"

    if ENABLE_INFERENCE_CACHE and all(
        path.is_file() for path in [raw_path, manifest_path, cache_marker_path]
    ):
        try:
            cache_marker = json.loads(cache_marker_path.read_text(encoding="utf-8"))

            # 저장된 raw_predictions_sha256/inference_manifest_sha256과 실제 파일 해시가
            # 일치하는지 확인해야 손상되거나 바뀐 캐시를 그대로 재사용하지 않는다.
            raw_hash_match = cache_marker.get("raw_predictions_sha256") == sha256_file(raw_path)
            manifest_hash_match = cache_marker.get("inference_manifest_sha256") == sha256_file(manifest_path)

            if (
                cache_marker.get("inference_signature") == inference_signature
                and raw_hash_match
                and manifest_hash_match
            ):
                raw_predictions_df = pd.read_csv(raw_path)
                inference_manifest_df = pd.read_csv(manifest_path)
                validate_model_inference_output(
                    model_name,
                    raw_predictions_df,
                    inference_manifest_df,
                    preprocessing_manifest,
                    mapping["kaggle_official_original_category_ids"],
                )
                print(f"Reused verified {model_name} inference cache: {raw_path}")
                return {
                    "model_name": model_name,
                    "raw_predictions_df": raw_predictions_df,
                    "inference_manifest_df": inference_manifest_df,
                    "raw_predictions_path": raw_path,
                    "inference_manifest_path": manifest_path,
                    "cache_marker_path": cache_marker_path,
                    "inference_signature": inference_signature,
                    "inference_seconds": float(cache_marker["inference_seconds"]),
                    "peak_gpu_memory_gb": float(cache_marker.get("peak_gpu_memory_gb", 0.0)),
                    "cache_reused": True,
                    "allowed_original_category_ids": sorted(
                        mapping["allowed_original_category_ids"]
                    ),
                }
        except (OSError, ValueError, KeyError, json.JSONDecodeError, RuntimeError):
            print(f"Ignored invalid {model_name} inference cache and reran inference.")

    release_inference_bundle(None)
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    bundle = None
    inference_started_wall_at = datetime.now()
    inference_started_perf = time.perf_counter()
    try:
        print(f"Loading {model_name}. GPU allocated before load: {current_gpu_memory_gb():.2f} GB")
        bundle = load_inference_bundle(model_name)
        prediction_rows = run_yolo_kaggle_inference(bundle, preprocessing_manifest)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        inference_seconds = time.perf_counter() - inference_started_perf
        peak_gpu_memory_gb = (
            float(torch.cuda.max_memory_allocated() / (1024 ** 3))
            if torch.cuda.is_available()
            else 0.0
        )
    finally:
        release_inference_bundle(bundle)

    raw_predictions_df = pd.DataFrame(prediction_rows, columns=PREDICTION_COLUMNS)
    if not raw_predictions_df.empty:
        raw_predictions_df = raw_predictions_df.sort_values(
            ["file_name", "rank"], ascending=[True, True]
        ).reset_index(drop=True)
    inference_manifest_df = build_inference_manifest(
        raw_predictions_df,
        preprocessing_manifest,
    )
    validate_model_inference_output(
        model_name,
        raw_predictions_df,
        inference_manifest_df,
        preprocessing_manifest,
        mapping["kaggle_official_original_category_ids"],
    )

    raw_predictions_df.to_csv(raw_path, index=False, encoding="utf-8-sig")
    inference_manifest_df.to_csv(manifest_path, index=False, encoding="utf-8-sig")
    cache_marker = {
        "created_at": datetime.now().isoformat(),
        "inference_started_at": inference_started_wall_at.isoformat(),
        "inference_signature": inference_signature,
        "contract": inference_contract,
        "inference_seconds": inference_seconds,
        "peak_gpu_memory_gb": peak_gpu_memory_gb,
        "detected_objects": len(raw_predictions_df),
        "raw_predictions_sha256": sha256_file(raw_path),
        "inference_manifest_sha256": sha256_file(manifest_path),
    }
    write_json_atomic(cache_marker_path, cache_marker)

    print(
        f"{model_name} completed: images={len(inference_manifest_df)}, "
        f"objects={len(raw_predictions_df)}, seconds={inference_seconds:.2f}, "
        f"peak_gpu={peak_gpu_memory_gb:.2f} GB"
    )
    return {
        "model_name": model_name,
        "raw_predictions_df": raw_predictions_df,
        "inference_manifest_df": inference_manifest_df,
        "raw_predictions_path": raw_path,
        "inference_manifest_path": manifest_path,
        "cache_marker_path": cache_marker_path,
        "inference_signature": inference_signature,
        "inference_seconds": inference_seconds,
        "peak_gpu_memory_gb": peak_gpu_memory_gb,
        "cache_reused": False,
        "allowed_original_category_ids": sorted(mapping["allowed_original_category_ids"]),
    }


MODEL_INFERENCE_OUTPUTS = {}
for model_name in tqdm(MODELS_TO_SUBMIT, desc="Three-model Kaggle inference", unit="model"):
    MODEL_INFERENCE_OUTPUTS[model_name] = execute_one_model_inference(
        model_name,
        preprocessing_manifest_df,
    )

submission_batch_id = (
    f"{datetime.now().strftime('%Y%m%d_%H%M%S')}_"
    f"{selection_signature}_{preprocessing_signature}"
)
model_inference_timing_table = pd.DataFrame([
    {
        "model": model_name,
        "images": len(output["inference_manifest_df"]),
        "detected_objects": len(output["raw_predictions_df"]),
        "selected_confidence": MODEL_RUN_CONFIGS[model_name]["selected_confidence"],
        "selected_top_k": MODEL_RUN_CONFIGS[model_name].get("selected_top_k"),
        "inference_seconds": output["inference_seconds"],
        "inference_minutes": output["inference_seconds"] / 60.0,
        "peak_gpu_memory_gb": output["peak_gpu_memory_gb"],
        "cache_reused": output["cache_reused"],
    }
    for model_name, output in MODEL_INFERENCE_OUTPUTS.items()
])
display(model_inference_timing_table)
show_stage_progress(11, "Run YOLO11s, YOLO11m, and YOLO12m inference", "DONE")

[Pipeline 11/14 |  79%] START: Run YOLO11s, YOLO11m, and YOLO12m inference


Three-model Kaggle inference:   0%|          | 0/3 [00:00<?, ?model/s]

Loading YOLO11s. GPU allocated before load: 0.03 GB


YOLO11s Kaggle inference:   0%|          | 0/842 [00:00<?, ?image/s]

YOLO11s completed: images=842, objects=3102, seconds=32.22, peak_gpu=1.08 GB
Loading YOLO11m. GPU allocated before load: 0.03 GB


...

YOLO11m completed: images=842, objects=3124, seconds=34.61, peak_gpu=1.86 GB
Loading YOLO12m. GPU allocated before load: 0.03 GB


YOLO12m Kaggle inference:   0%|          | 0/842 [00:00<?, ?image/s]

YOLO12m completed: images=842, objects=2959, seconds=37.82, peak_gpu=2.52 GB


,model,images,detected_objects,selected_confidence,selected_top_k,inference_seconds,inference_minutes,peak_gpu_memory_gb,cache_reused
0,YOLO11s,842,3102,0.001,4,32.217459,0.536958,1.075649,False
1,YOLO11m,842,3124,0.001,4,34.605309,0.576755,1.856948,False
2,YOLO12m,842,2959,0.001,4,37.817344,0.630289,2.523273,False


[Pipeline 11/14 |  79%] DONE: Run YOLO11s, YOLO11m, and YOLO12m inference


# 7. 예측 결과 시각화

원본 이미지 위에 모델별 BBox를 그려 padding 제거와 좌표 역변환이 올바른지 확인한다. 같은 이미지의 세 모델 결과를 나란히 비교해 오탐과 미탐 차이도 확인한다.

In [28]:
show_stage_progress(12, "Create prediction previews for three models", "START")


def draw_prediction_preview(image_path, prediction_df):
    """원본 이미지에 category ID와 confidence를 표시한다."""
    with Image.open(image_path) as image:
        canvas = ImageOps.exif_transpose(image).convert("RGB")
    drawer = ImageDraw.Draw(canvas)
    line_width = max(2, round(min(canvas.size) / 300))
    for row in prediction_df.itertuples(index=False):
        box = [row.x_min, row.y_min, row.x_max, row.y_max]
        drawer.rectangle(box, outline=(255, 40, 40), width=line_width)
        label = f"{int(row.category_id)} {float(row.score):.2f}"
        text_position = (max(0, row.x_min), max(0, row.y_min - 14))
        drawer.text(text_position, label, fill=(255, 40, 40))
    return canvas


PREVIEW_PATHS_BY_MODEL = {}
preview_count = min(VISUALIZATION_IMAGE_COUNT, len(active_test_image_paths))
for model_name in tqdm(MODELS_TO_SUBMIT, desc="Create model previews", unit="model"):
    raw_predictions_df = MODEL_INFERENCE_OUTPUTS[model_name]["raw_predictions_df"]
    fig, axes = plt.subplots(preview_count, 1, figsize=(12, 5 * preview_count))
    if preview_count == 1:
        axes = [axes]
    for axis, image_path in zip(axes, active_test_image_paths[:preview_count]):
        image_predictions = raw_predictions_df[
            raw_predictions_df["file_name"] == image_path.name
        ]
        preview = draw_prediction_preview(image_path, image_predictions)
        axis.imshow(preview)
        axis.set_title(
            f"{model_name} | {image_path.name} | detections={len(image_predictions)}"
        )
        axis.axis("off")
    plt.tight_layout()
    preview_path = FIGURE_DIR / (
        f"kaggle_prediction_preview_{model_name.lower()}_{submission_batch_id}.png"
    )
    plt.savefig(preview_path, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    PREVIEW_PATHS_BY_MODEL[model_name] = preview_path
    print(f"{model_name} prediction preview: {preview_path}")

show_stage_progress(12, "Create prediction previews for three models", "DONE")


Output hidden; open in https://colab.research.google.com to view.

# 8. Kaggle 제출 파일 생성

Kaggle 제출 형식에 맞춰 모델별 CSV를 생성한다.

- 컬럼: `annotation_id, image_id, category_id, bbox_x, bbox_y, bbox_w, bbox_h, score`
- `image_id`: `inference_images` 파일명에서 복원
- `category_id`: 모델 class 0..55를 원본 category ID로 변환
- BBox: 원본 이미지 기준 `[x, y, width, height]`
- 적용 순서: 56-class 필터 → confidence → score 정렬 → 선택된 Top-K
- 모델/stage/confidence/Top-K: Kaggle-domain 평가 summary 사용

`smoke`에서는 일부 이미지만 검증하고, `full`에서 전체 842장 제출 CSV를 생성한다.

In [29]:
show_stage_progress(13, "Build and validate three submission tables", "START")

def build_competition_submission(
    predictions_df,
    preprocessing_manifest,
    image_id_mapping,
):
    """공통 원시 예측을 대회가 요구하는 객체별 8개 컬럼으로 변환한다."""
    required_prediction_columns = {
        "file_name", "category_id", "score", "x", "y", "width", "height", "rank",
    }
    missing_prediction_columns = required_prediction_columns - set(predictions_df.columns)
    if missing_prediction_columns:
        raise ValueError(
            f"Raw prediction columns missing: {sorted(missing_prediction_columns)}"
        )

    dimensions = preprocessing_manifest[[
        "file_name", "source_width", "source_height",
    ]].copy()
    if dimensions["file_name"].duplicated().any():
        raise RuntimeError("Duplicate filenames were found in the preprocessing manifest.")

    if predictions_df.empty:
        empty_df = pd.DataFrame(columns=KAGGLE_SUBMISSION_COLUMNS)
        dtype_spec = {
            "annotation_id": np.int64,
            "image_id": np.int64,
            "category_id": np.int64,
            "bbox_x": float,
            "bbox_y": float,
            "bbox_w": float,
            "bbox_h": float,
            "score": float,
        }
        return empty_df.astype(dtype_spec)

    work = predictions_df.merge(
        image_id_mapping,
        on="file_name",
        how="left",
        validate="many_to_one",
    ).merge(
        dimensions,
        on="file_name",
        how="left",
        validate="many_to_one",
    )
    if work[["image_id", "source_width", "source_height"]].isna().any().any():
        unknown_files = work.loc[
            work["image_id"].isna() | work["source_width"].isna(),
            "file_name",
        ].unique().tolist()
        raise RuntimeError(f"Predictions could not be mapped to test images: {unknown_files[:10]}")

    work = work.sort_values(["image_id", "rank"], ascending=[True, True]).reset_index(drop=True)
    submission_df = pd.DataFrame({
        "annotation_id": np.arange(1, len(work) + 1, dtype=np.int64),
        "image_id": work["image_id"].astype(np.int64),
        "category_id": work["category_id"].astype(np.int64),
        "bbox_x": work["x"].astype(float),
        "bbox_y": work["y"].astype(float),
        "bbox_w": work["width"].astype(float),
        "bbox_h": work["height"].astype(float),
        "score": work["score"].astype(float),
    })
    return submission_df[KAGGLE_SUBMISSION_COLUMNS]


def validate_competition_submission(
    submission_df,
    preprocessing_manifest,
    image_id_mapping,
    allowed_category_ids,
    require_all_test_images,
):
    """공식 컬럼·ID·원본 좌표·confidence 계약을 검사한다. Top-K는 모델별 Validation 정책으로 이미 적용된다."""
    exact_columns = submission_df.columns.tolist() == list(KAGGLE_SUBMISSION_COLUMNS)
    required_test_count = EXPECTED_KAGGLE_TEST_IMAGES if require_all_test_images else len(
        preprocessing_manifest
    )
    active_mapping = image_id_mapping[
        image_id_mapping["file_name"].isin(preprocessing_manifest["file_name"])
    ].copy()
    active_mapping["image_id"] = active_mapping["image_id"].astype(np.int64)

    dimension_table = active_mapping.merge(
        preprocessing_manifest[["file_name", "source_width", "source_height"]],
        on="file_name",
        how="inner",
        validate="one_to_one",
    )
    dimension_by_image_id = dimension_table.set_index("image_id")[[
        "source_width", "source_height",
    ]]

    no_missing_values = not submission_df.isna().any().any()
    finite_numeric_values = True
    if not submission_df.empty:
        numeric_columns = KAGGLE_SUBMISSION_COLUMNS
        numeric_values = submission_df[numeric_columns].to_numpy(dtype=float)
        finite_numeric_values = bool(np.isfinite(numeric_values).all())

    expected_annotation_ids = list(range(1, len(submission_df) + 1))
    annotation_ids_valid = (
        submission_df["annotation_id"].astype(int).tolist() == expected_annotation_ids
        if exact_columns else False
    )
    allowed_image_ids = set(active_mapping["image_id"].astype(np.int64))
    submitted_image_ids = (
        set(submission_df["image_id"].astype(np.int64)) if exact_columns else set()
    )
    image_ids_valid = submitted_image_ids <= allowed_image_ids
    submitted_category_ids = (
        set(submission_df["category_id"].astype(np.int64)) if exact_columns else set()
    )
    category_ids_valid = submitted_category_ids <= set(map(int, allowed_category_ids))

    scores_valid = True
    boxes_positive = True
    boxes_inside_original_images = True
    maximum_objects = 0
    if exact_columns and not submission_df.empty:
        scores_valid = bool(submission_df["score"].between(0.0, 1.0, inclusive="both").all())
        boxes_positive = bool(
            (submission_df["bbox_x"] >= 0).all()
            and (submission_df["bbox_y"] >= 0).all()
            and (submission_df["bbox_w"] > 0).all()
            and (submission_df["bbox_h"] > 0).all()
        )
        checked = submission_df.merge(
            dimension_by_image_id,
            left_on="image_id",
            right_index=True,
            how="left",
            validate="many_to_one",
        )
        tolerance = 1e-4
        boxes_inside_original_images = bool(
            checked[["source_width", "source_height"]].notna().all().all()
            and (checked["bbox_x"] + checked["bbox_w"] <= checked["source_width"] + tolerance).all()
            and (checked["bbox_y"] + checked["bbox_h"] <= checked["source_height"] + tolerance).all()
        )
        maximum_objects = int(submission_df.groupby("image_id").size().max())

    # 이미지당 최대 검출 객체 수가 MAX_PILLS_PER_IMAGE 제한을 넘지 않는지 검증한다.
    per_image_count_valid = (
        MAX_PILLS_PER_IMAGE is None or maximum_objects <= MAX_PILLS_PER_IMAGE
    )

    checks = [
        {
            "check": "Exact competition column order",
            "status": "PASS" if exact_columns else "FAIL",
            "evidence": submission_df.columns.tolist(),
        },
        {
            "check": "Expected test image ID mapping count",
            "status": "PASS" if len(active_mapping) == required_test_count else "FAIL",
            "evidence": len(active_mapping),
        },
        {
            "check": "Unique numeric image IDs",
            "status": "PASS" if active_mapping["image_id"].is_unique else "FAIL",
            "evidence": active_mapping["image_id"].nunique(),
        },
        {
            "check": "At least one detected object",
            "status": "PASS" if len(submission_df) > 0 else "FAIL",
            "evidence": len(submission_df),
        },
        {
            "check": "No missing or non-finite values",
            "status": "PASS" if no_missing_values and finite_numeric_values else "FAIL",
            "evidence": len(submission_df),
        },
        {
            "check": "Sequential unique annotation IDs",
            "status": "PASS" if annotation_ids_valid else "FAIL",
            "evidence": len(expected_annotation_ids),
        },
        {
            "check": "Known test image IDs only",
            "status": "PASS" if image_ids_valid else "FAIL",
            "evidence": len(submitted_image_ids),
        },
        {
            "check": "Allowed original category IDs only",
            "status": "PASS" if category_ids_valid else "FAIL",
            "evidence": len(submitted_category_ids),
        },
        {
            "check": "Scores are within zero and one",
            "status": "PASS" if scores_valid else "FAIL",
            "evidence": float(submission_df["score"].min()) if len(submission_df) else None,
        },
        {
            "check": "Positive BBoxes inside original images",
            "status": "PASS" if boxes_positive and boxes_inside_original_images else "FAIL",
            "evidence": "original-image xywh",
        },
        {
            "check": "Per-image prediction count recorded",
            "status": "PASS" if per_image_count_valid else "FAIL",
            "evidence": maximum_objects,
        },
    ]
    return pd.DataFrame(checks)


SUBMISSION_TABLES_BY_MODEL = {}
VALIDATION_TABLES_BY_MODEL = {}
validation_summary_frames = []

for model_name in tqdm(MODELS_TO_SUBMIT, desc="Validate model submissions", unit="model"):
    output = MODEL_INFERENCE_OUTPUTS[model_name]
    submission_df = build_competition_submission(
        output["raw_predictions_df"],
        preprocessing_manifest_df,
        test_image_id_mapping_df,
    )
    validation_table = validate_competition_submission(
        submission_df,
        preprocessing_manifest_df,
        test_image_id_mapping_df,
        output["allowed_original_category_ids"],
        require_all_test_images=(RUN_MODE == "full"),
    )
    SUBMISSION_TABLES_BY_MODEL[model_name] = submission_df
    VALIDATION_TABLES_BY_MODEL[model_name] = validation_table
    validation_summary_frames.append(validation_table.assign(model=model_name))

    print(f"{model_name} submission preview:")
    display(submission_df.head(10))
    display(validation_table)

ALL_MODEL_VALIDATION_TABLE = pd.concat(
    validation_summary_frames,
    ignore_index=True,
)[["model", "check", "status", "evidence"]]
print(f"Competition metric: {COMPETITION_METRIC}")
display(ALL_MODEL_VALIDATION_TABLE)
show_stage_progress(13, "Build and validate three submission tables", "DONE")

[Pipeline 13/14 |  93%] START: Build and validate three submission tables


Validate model submissions:   0%|          | 0/3 [00:00<?, ?model/s]

YOLO11s submission preview:


,annotation_id,image_id,category_id,bbox_x,bbox_y,bbox_w,bbox_h,score
0,1,1,1900,156.231120,252.083903,205.630697,125.802409,0.977491
1,2,1,24850,171.748413,739.647949,180.448893,293.964844,0.965639
2,3,1,16551,553.837728,71.676839,403.160482,409.026937,0.960675
3,4,1,27926,599.943359,669.648763,252.844727,485.105957,0.915665
4,5,3,1900,140.265177,241.218913,199.523275,129.729980,0.973431
5,6,3,24850,138.293213,699.002604,184.500488,300.193197,0.967699
6,7,3,16551,527.133260,63.435465,391.089559,398.600179,0.960087
7,8,3,27926,570.569661,625.440755,258.041992,491.798340,0.927807
8,9,4,29345,116.424540,150.390218,276.391622,419.800944,0.973012
9,10,4,1900,683.944987,806.859212,131.856445,209.500651,0.970319


,check,status,evidence
0,Exact competition column order,PASS,"[annotation_id, image_id, category_id, bbox_x,..."
1,Expected test image ID mapping count,PASS,842
2,Unique numeric image IDs,PASS,842
3,At least one detected object,PASS,3102
4,No missing or non-finite values,PASS,3102
5,Sequential unique annotation IDs,PASS,3102
6,Known test image IDs only,PASS,842
7,Allowed original category IDs only,PASS,56
8,Scores are within zero and one,PASS,0.001
9,Positive BBoxes inside original images,PASS,original-image xywh


...

,annotation_id,image_id,category_id,bbox_x,bbox_y,bbox_w,bbox_h,score
0,1,1,27926,598.185384,667.377441,256.416178,491.147786,0.996392
1,2,1,1900,157.812419,251.206828,203.612549,125.571615,0.988284
2,3,1,24850,170.856038,739.789876,182.077230,292.657552,0.981200
3,4,1,16551,552.744466,72.266846,403.139160,404.634766,0.967696
4,5,3,27926,570.677572,624.559814,260.751953,493.563965,0.998601
5,6,3,1900,140.925557,240.536296,198.811178,129.216634,0.992018
6,7,3,24850,137.258443,701.307943,188.603699,297.580892,0.983719
7,8,3,16551,526.742228,64.005005,392.039673,397.396647,0.974485
8,9,4,1900,683.810465,806.070312,130.284668,209.037760,0.986627
9,10,4,29345,118.219849,150.970785,273.650391,418.801351,0.981983


,check,status,evidence
0,Exact competition column order,PASS,"[annotation_id, image_id, category_id, bbox_x,..."
1,Expected test image ID mapping count,PASS,842
2,Unique numeric image IDs,PASS,842
3,At least one detected object,PASS,2959
4,No missing or non-finite values,PASS,2959
5,Sequential unique annotation IDs,PASS,2959
6,Known test image IDs only,PASS,842
7,Allowed original category IDs only,PASS,56
8,Scores are within zero and one,PASS,0.001001
9,Positive BBoxes inside original images,PASS,original-image xywh


Competition metric: mAP@[0.75:0.95]


,model,check,status,evidence
0,YOLO11s,Exact competition column order,PASS,"[annotation_id, image_id, category_id, bbox_x,..."
1,YOLO11s,Expected test image ID mapping count,PASS,842
2,YOLO11s,Unique numeric image IDs,PASS,842
3,YOLO11s,At least one detected object,PASS,3102
4,YOLO11s,No missing or non-finite values,PASS,3102
5,YOLO11s,Sequential unique annotation IDs,PASS,3102
6,YOLO11s,Known test image IDs only,PASS,842
7,YOLO11s,Allowed original category IDs only,PASS,56
8,YOLO11s,Scores are within zero and one,PASS,0.001
9,YOLO11s,Positive BBoxes inside original images,PASS,original-image xywh


[Pipeline 13/14 |  93%] DONE: Build and validate three submission tables


In [30]:
show_stage_progress(14, "Export three CSV files and sync final artifacts", "START")

failed_validation = ALL_MODEL_VALIDATION_TABLE[
    ALL_MODEL_VALIDATION_TABLE["status"] != "PASS"
]
if not failed_validation.empty:
    failed_pairs = failed_validation[["model", "check"]].to_dict("records")
    raise RuntimeError(f"Submission validation failed: {failed_pairs}")

VALIDATION_SUMMARY_PATH = REPORT_DIR / f"submission_checks_{submission_batch_id}.csv"
ALL_MODEL_VALIDATION_TABLE.to_csv(
    VALIDATION_SUMMARY_PATH,
    index=False,
    encoding="utf-8-sig",
)

SUBMISSION_PATHS_BY_MODEL = {}
VALIDATION_REPORT_PATHS_BY_MODEL = {}

if RUN_MODE == "smoke":
    print("IntegratedDataset.tar smoke inference completed successfully.")
    print("Set RUN_MODE='full', restart the runtime, and run all cells to create three CSV files.")
else:
    if len(preprocessing_manifest_df) != EXPECTED_KAGGLE_TEST_IMAGES:
        raise RuntimeError("Full submission requires all Kaggle test images.")

    for model_name in tqdm(MODELS_TO_SUBMIT, desc="Write Kaggle CSV files", unit="model"):
        output = MODEL_INFERENCE_OUTPUTS[model_name]
        submission_df = SUBMISSION_TABLES_BY_MODEL[model_name]
        inference_manifest_df = output["inference_manifest_df"]
        if inference_manifest_df["file_name"].nunique() != EXPECTED_KAGGLE_TEST_IMAGES:
            raise RuntimeError(f"{model_name} did not infer all test images.")

        submission_id = f"{model_name.lower()}_{submission_batch_id}"
        submission_path = SUBMISSION_DIR / f"submission_{submission_id}.csv"
        submission_df.to_csv(
            submission_path,
            index=False,
            encoding="utf-8",
            float_format="%.6f",
        )
        saved_submission_df = pd.read_csv(submission_path)
        if saved_submission_df.columns.tolist() != KAGGLE_SUBMISSION_COLUMNS:
            raise RuntimeError(f"{model_name} saved submission columns changed unexpectedly.")
        if len(saved_submission_df) != len(submission_df):
            raise RuntimeError(f"{model_name} saved submission row count changed unexpectedly.")

        predicted_image_count = int(submission_df["image_id"].nunique())
        validation_report = {
            "pipeline_version": PIPELINE_VERSION,
            "created_at": datetime.now().isoformat(),
            "competition_metric": COMPETITION_METRIC,
            "model_contract_path": FINAL_CONFIG_PATH,
            "model_name": model_name,
            "experiment_id": MODEL_RUN_CONFIGS[model_name]["experiment_id"],
            "checkpoint_sha256": MODEL_RUN_CONFIGS[model_name]["checkpoint_sha256"],
            "selected_confidence": MODEL_RUN_CONFIGS[model_name]["selected_confidence"],
            "selected_top_k": MODEL_RUN_CONFIGS[model_name].get("selected_top_k"),
            "use_tta": USE_TTA,
            "preprocessing_signature": preprocessing_signature,
            "raw_predictions_path": output["raw_predictions_path"],
            "image_id_mapping_path": TEST_IMAGE_ID_MAPPING_PATH,
            "submission_path": submission_path,
            "submission_sha256": sha256_file(submission_path),
            "submission_columns": KAGGLE_SUBMISSION_COLUMNS,
            "coordinate_format": "original-image xywh",
            "test_images": EXPECTED_KAGGLE_TEST_IMAGES,
            "images_with_predictions": predicted_image_count,
            "images_without_predictions": EXPECTED_KAGGLE_TEST_IMAGES - predicted_image_count,
            "detected_objects": len(submission_df),
            "checks": VALIDATION_TABLES_BY_MODEL[model_name].to_dict("records"),
        }
        validation_report_path = REPORT_DIR / f"submission_validation_{submission_id}.json"
        write_json_atomic(validation_report_path, validation_report)
        SUBMISSION_PATHS_BY_MODEL[model_name] = submission_path
        VALIDATION_REPORT_PATHS_BY_MODEL[model_name] = validation_report_path
        print(f"{model_name} submission: {submission_path}")


# Kaggle 첫 제출용으로 전체 1등 단일 모델 CSV를 명확한 이름으로 하나 더 만든다.
RECOMMENDED_SUBMISSION_PATH = None
if RUN_MODE == "full":
    recommended_source = Path(SUBMISSION_PATHS_BY_MODEL[RECOMMENDED_MODEL])
    RECOMMENDED_SUBMISSION_PATH = (
        SUBMISSION_DIR
        / f"submission_RECOMMENDED_{RECOMMENDED_MODEL.lower()}_{submission_batch_id}.csv"
    )
    shutil.copy2(recommended_source, RECOMMENDED_SUBMISSION_PATH)
    if sha256_file(recommended_source) != sha256_file(RECOMMENDED_SUBMISSION_PATH):
        raise RuntimeError("Recommended submission copy SHA256 mismatch.")
    print(
        f"RECOMMENDED FIRST KAGGLE SUBMISSION: {RECOMMENDED_MODEL} -> "
        f"{RECOMMENDED_SUBMISSION_PATH}"
    )


pipeline_seconds = time.perf_counter() - pipeline_timer_started_perf
pipeline_finished_at = datetime.now()

KAGGLE_TIMING_JSON_PATH = REPORT_DIR / f"kaggle_timing_{submission_batch_id}.json"
KAGGLE_TIMING_CSV_PATH = REPORT_DIR / f"kaggle_timing_{submission_batch_id}.csv"
kaggle_timing_table = model_inference_timing_table.copy()
kaggle_timing_table["run_mode"] = RUN_MODE
kaggle_timing_table["preprocessing_seconds_shared"] = preprocessing_seconds
kaggle_timing_table["pipeline_seconds_total"] = pipeline_seconds
kaggle_timing_table.to_csv(
    KAGGLE_TIMING_CSV_PATH,
    index=False,
    encoding="utf-8-sig",
)
kaggle_timing_record = {
    "schema_version": 2,
    "pipeline_version": PIPELINE_VERSION,
    "run_mode": RUN_MODE,
    "models": list(MODELS_TO_SUBMIT),
    "recommended_model": RECOMMENDED_MODEL,
    "recommended_competition_mAP": RECOMMENDED_COMPETITION_MAP,
    "images_per_model": len(active_test_image_paths),
    "storage_mode": "colab_local_processing_drive_final_outputs",
    "input_staging_mode": input_staging_mode,
    "preprocessing_cache_reused": bool(reuse_preprocessing),
    "pipeline_started_at": pipeline_timer_started_at.isoformat(),
    "pipeline_finished_at": pipeline_finished_at.isoformat(),
    "preprocessing_seconds": preprocessing_seconds,
    "pipeline_seconds": pipeline_seconds,
    "model_inference": model_inference_timing_table.to_dict("records"),
    "submission_paths": SUBMISSION_PATHS_BY_MODEL,
}
write_json_atomic(KAGGLE_TIMING_JSON_PATH, kaggle_timing_record)
display(kaggle_timing_table)

# 전처리 이미지는 로컬에만 두고 재현·제출에 필요한 작은 산출물만 Drive에 보존한다.
artifact_targets = [
    (FINAL_CONFIG_PATH, DRIVE_FINAL_CONFIG_DIR),
    (LOCAL_STAGING_MANIFEST_PATH, DRIVE_PREPROCESSING_MANIFEST_DIR),
    (TEST_IMAGE_ID_MAPPING_PATH, DRIVE_PREPROCESSING_MANIFEST_DIR),
    (PREPROCESSING_MANIFEST_PATH, DRIVE_PREPROCESSING_MANIFEST_DIR),
    (VALIDATION_SUMMARY_PATH, DRIVE_REPORT_DIR),
    (KAGGLE_TIMING_JSON_PATH, DRIVE_REPORT_DIR),
    (KAGGLE_TIMING_CSV_PATH, DRIVE_REPORT_DIR),
]
if RUN_MODE == "full" and RECOMMENDED_SUBMISSION_PATH is not None:
    artifact_targets.append(
        (RECOMMENDED_SUBMISSION_PATH, DRIVE_SUBMISSION_DIR)
    )

for model_name in MODELS_TO_SUBMIT:
    output = MODEL_INFERENCE_OUTPUTS[model_name]
    artifact_targets.extend([
        (output["raw_predictions_path"], DRIVE_PREDICTION_DIR),
        (output["inference_manifest_path"], DRIVE_PREDICTION_DIR),
        (output["cache_marker_path"], DRIVE_PREDICTION_DIR),
        (PREVIEW_PATHS_BY_MODEL[model_name], DRIVE_FIGURE_DIR),
    ])
    if RUN_MODE == "full":
        artifact_targets.extend([
            (SUBMISSION_PATHS_BY_MODEL[model_name], DRIVE_SUBMISSION_DIR),
            (VALIDATION_REPORT_PATHS_BY_MODEL[model_name], DRIVE_REPORT_DIR),
        ])

drive_artifact_rows = []
for local_artifact_path, drive_directory in tqdm(
    artifact_targets,
    desc="Copy final artifacts to Drive",
    unit="file",
):
    drive_artifact_path = persist_artifact_to_drive(local_artifact_path, drive_directory)
    drive_artifact_rows.append({
        "local_path": str(local_artifact_path),
        "drive_path": str(drive_artifact_path),
        "sha256": sha256_file(local_artifact_path),
        "size_bytes": Path(local_artifact_path).stat().st_size,
    })

DRIVE_SYNC_MANIFEST_PATH = REPORT_DIR / f"drive_sync_{submission_batch_id}.json"
write_json_atomic(DRIVE_SYNC_MANIFEST_PATH, {
    "created_at": datetime.now().isoformat(),
    "storage_mode": "colab_local_processing_drive_final_outputs",
    "preprocessed_images_copied_to_drive": False,
    "models": list(MODELS_TO_SUBMIT),
    "artifacts": drive_artifact_rows,
})
drive_sync_path = persist_artifact_to_drive(DRIVE_SYNC_MANIFEST_PATH, DRIVE_REPORT_DIR)

print("Final artifacts were copied to Google Drive.")
print(f"Drive sync manifest: {drive_sync_path}")
if RUN_MODE == "full":
    recommended_drive_path = next(
        row["drive_path"] for row in drive_artifact_rows
        if RECOMMENDED_SUBMISSION_PATH is not None
        and Path(row["local_path"]) == Path(RECOMMENDED_SUBMISSION_PATH)
    )
    print("=" * 80)
    print(
        f"FIRST KAGGLE SUBMISSION RECOMMENDATION: {RECOMMENDED_MODEL} "
        f"({COMPETITION_METRIC}={RECOMMENDED_COMPETITION_MAP:.6f})"
    )
    print(f"Recommended CSV: {recommended_drive_path}")
    print("Other architecture-winner CSV files (optional additional submissions):")
    for model_name in MODELS_TO_SUBMIT:
        local_submission = Path(SUBMISSION_PATHS_BY_MODEL[model_name])
        drive_submission = next(
            row["drive_path"] for row in drive_artifact_rows
            if Path(row["local_path"]) == local_submission
        )
        print(f"- {model_name}: {drive_submission}")

show_stage_progress(14, "Export three CSV files and sync final artifacts", "DONE")

[Pipeline 14/14 | 100%] START: Export three CSV files and sync final artifacts


Write Kaggle CSV files:   0%|          | 0/3 [00:00<?, ?model/s]

YOLO11s submission: /content/baby_kangaroo_kaggle_submission_v30/results/submissions/submission_yolo11s_20260819_052624_605a510f5e79d59d_14ebf6bb7189a8b6.csv
YOLO11m submission: /content/baby_kangaroo_kaggle_submission_v30/results/submissions/submission_yolo11m_20260819_052624_605a510f5e79d59d_14ebf6bb7189a8b6.csv
YOLO12m submission: /content/baby_kangaroo_kaggle_submission_v30/results/submissions/submission_yolo12m_20260819_052624_605a510f5e79d59d_14ebf6bb7189a8b6.csv
RECOMMENDED FIRST KAGGLE SUBMISSION: YOLO12m -> /content/baby_kangaroo_kaggle_submission_v30/results/submissions/submission_RECOMMENDED_yolo12m_20260819_052624_605a510f5e79d59d_14ebf6bb7189a8b6.csv


,model,images,detected_objects,selected_confidence,selected_top_k,inference_seconds,inference_minutes,peak_gpu_memory_gb,cache_reused,run_mode,preprocessing_seconds_shared,pipeline_seconds_total
0,YOLO11s,842,3102,0.001,4,32.217459,0.536958,1.075649,False,full,246.648365,386.111629
1,YOLO11m,842,3124,0.001,4,34.605309,0.576755,1.856948,False,full,246.648365,386.111629
2,YOLO12m,842,2959,0.001,4,37.817344,0.630289,2.523273,False,full,246.648365,386.111629


Copy final artifacts to Drive:   0%|          | 0/26 [00:00<?, ?file/s]

Copied to Colab local storage: integrated_submission_config_605a510f5e79d59d.json
Copied to Colab local storage: local_input_staging_manifest.csv
Copied to Colab local storage: test_image_id_mapping.csv
Copied to Colab local storage: preprocessing_manifest.csv
Copied to Colab local storage: submission_checks_20260819_052624_605a510f5e79d59d_14ebf6bb7189a8b6.csv
Copied to Colab local storage: kaggle_timing_20260819_052624_605a510f5e79d59d_14ebf6bb7189a8b6.json
Copied to Colab local storage: kaggle_timing_20260819_052624_605a510f5e79d59d_14ebf6bb7189a8b6.csv
Copied to Colab local storage: submission_RECOMMENDED_yolo12m_20260819_052624_605a510f5e79d59d_14ebf6bb7189a8b6.csv
Copied to Colab local storage: raw_predictions_yolo11s_6ab69fac43eb97ce.csv
Copied to Colab local storage: inference_manifest_yolo11s_6ab69fac43eb97ce.csv
Copied to Colab local storage: inference_cache_yolo11s_6ab69fac43eb97ce.json
Copied to Colab local storage: kaggle_prediction_preview_yolo11s_20260819_052624_605a510f

## 실행 순서

1. Kaggle-domain 평가 결과와 모델 체크포인트를 준비한다.
2. `RUN_MODE="smoke"`로 경로, 전처리, 추론, 제출 형식을 확인한다.
3. 문제가 없으면 `RUN_MODE="full"`로 전체 842장을 처리한다.
4. 생성된 제출 CSV의 행 수, 컬럼, `image_id`, BBox 범위를 마지막으로 확인한다.